# GAIN / WGAN-FWAL / KD 결측치 대체 실험 — Light & Heavy DB

본 노트북은 **5개 Light 데이터셋 × 5개 결측률** 및 **2개 Heavy 데이터셋**에 대해
**GAIN**, **WGAN-FWAL**, **Knowledge Distillation(2단계)** 세 모델을 실행하고
**다중 성능 지표(RMSE, MAE, AUROC, PredAUC)** 및 **고급 평가(FLOPs, Latency, FID, Diversity)** 를 비교합니다.

| 데이터셋 | 결측률 |
|---|---|
| breast_cancer, spam, credit, wine, student | 0.05, 0.1, 0.2, 0.3, 0.5 |
| HIGGS, Criteo (Heavy) | 0.1, 0.2, 0.3 |

---
### 목차
1. 패키지 설치 및 임포트
2. 유틸리티 함수 (`utils.py`)
3. 성능 평가 함수 (`evaluation.py` + `advanced_eval.py`)
4. Light 데이터 로더 (`lightDBloader.py`)
5. Heavy 데이터 로더 (`heavyDBloader.py`)
6. 모델 정의 — GAIN
7. 모델 정의 — WGAN-FWAL
8. Knowledge Distillation — 2단계
9. Light DB 메인 실험 루프
10. Heavy DB 메인 실험 루프
11. 결과 시각화 (Light + Heavy 통합)
12. 고급 평가 — FLOPs / Latency / FID / Diversity


---
## 1. 패키지 설치 및 임포트

In [ ]:
# 필요한 패키지 설치 (최초 1회)
!pip install torch --index-url https://download.pytorch.org/whl/cu126 -q
!pip install pandas matplotlib seaborn scikit-learn ucimlrepo tqdm scipy -q


In [ ]:
import os, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from scipy import stats
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score
from torch.utils.data import Dataset, DataLoader
from ucimlrepo import fetch_ucirepo

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device    : {DEVICE}')

# ── AMP / GPU / Tensor Core 설정 ─────────────────────────
from torch.cuda.amp import autocast, GradScaler

USE_AMP = torch.cuda.is_available()   # GPU 없으면 자동 비활성화

if torch.cuda.is_available():
    # cuDNN auto-tuner: 고정 입력 크기에서 최적 알고리즘 선택
    torch.backends.cudnn.benchmark = True
    # Ampere(A6000) 이상에서 TF32 Tensor Core 사용 (FP32 대비 ~2x)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32       = True
    print(f'GPU  : {torch.cuda.get_device_name(0)}')
    print(f'AMP  : float16 mixed precision ON')
    print(f'TF32 : Tensor Core ON (Ampere+ GPU)')
else:
    print('GPU 없음 — CPU 모드, AMP 비활성화')


---
## 2. 유틸리티 함수 (`utils.py`)


In [ ]:
# ════════════════════════════════════════════════════════
# utils.py
# ════════════════════════════════════════════════════════

def normalization(data, parameters=None):
    """MinMax normalization to [0, 1]."""
    _, dim = data.shape
    norm_data = data.copy()

    if parameters is None:
        min_val = np.zeros(dim)
        max_val = np.zeros(dim)
        for i in range(dim):
            min_val[i] = np.nanmin(norm_data[:, i])
            norm_data[:, i] = norm_data[:, i] - min_val[i]
            max_val[i] = np.nanmax(norm_data[:, i])
            norm_data[:, i] = norm_data[:, i] / (max_val[i] + 1e-6)
        norm_parameters = {'min_val': min_val, 'max_val': max_val}
    else:
        min_val = parameters['min_val']
        max_val = parameters['max_val']
        for i in range(dim):
            norm_data[:, i] = norm_data[:, i] - min_val[i]
            norm_data[:, i] = norm_data[:, i] / (max_val[i] + 1e-6)
        norm_parameters = parameters

    return norm_data, norm_parameters


def renormalization(norm_data, norm_parameters):
    """역정규화: [0,1] → 원본 스케일."""
    min_val = norm_parameters['min_val']
    max_val = norm_parameters['max_val']
    _, dim = norm_data.shape
    renorm_data = norm_data.copy()
    for i in range(dim):
        renorm_data[:, i] = renorm_data[:, i] * (max_val[i] + 1e-6)
        renorm_data[:, i] = renorm_data[:, i] + min_val[i]
    return renorm_data


def rounding(imputed_data, data_x):
    """범주형 변수에 대해 반올림 처리."""
    _, dim = data_x.shape
    rounded_data = imputed_data.copy()
    for i in range(dim):
        temp = data_x[~np.isnan(data_x[:, i]), i]
        if len(np.unique(temp)) < 20:
            rounded_data[:, i] = np.round(rounded_data[:, i])
    return rounded_data


def rmse_loss(ori_data, imputed_data, data_m):
    """결측 위치에 대한 RMSE 계산."""
    ori_data, norm_parameters = normalization(ori_data)
    imputed_data, _ = normalization(imputed_data, norm_parameters)
    nominator   = np.sum(((1 - data_m) * ori_data - (1 - data_m) * imputed_data) ** 2)
    denominator = np.sum(1 - data_m)
    return np.sqrt(nominator / float(denominator))


def binary_sampler(p, rows, cols):
    """베르누이 샘플 행렬 생성."""
    return (np.random.uniform(0., 1., size=[rows, cols]) < p).astype(float)


def uniform_sampler(low, high, rows, cols):
    """균등분포 샘플 행렬 생성."""
    return np.random.uniform(low, high, size=[rows, cols])


def sample_batch_index(total, batch_size):
    """미니배치 인덱스 샘플링."""
    return np.random.permutation(total)[:batch_size]


print("✅ utils 함수 정의 완료")

---
## 3. 성능 평가 함수 (`evaluation.py` + `advanced_eval.py`)

### 3-1. 기본 지표
| 함수 | 설명 |
|---|---|
| `rmse_loss` | 결측 위치 RMSE (정규화 후) |
| `mae_loss` | 결측 위치 MAE (정규화 후) |
| `auroc_score` | 결측 위치 실제값 이진화 → AUROC |
| `prediction_performance` | 마지막 컬럼 레이블 기준 5-fold CV AUC |
| `evaluate_all` | 위 4개 지표 딕셔너리로 반환 |

### 3-2. 고급 지표 (advanced_eval)
| 함수 | 설명 |
|---|---|
| `count_flops` / `count_parameters` | Linear layer 기준 FLOPs·파라미터 수 |
| `measure_latency` | 평균 추론 시간 (ms, 20회 반복) |
| `record_gradient_magnitudes` | 레이어별 gradient norm (학습 중 수집) |
| `compute_fid` | 간소화 FID (평균 벡터 차이 + 공분산 trace) |
| `diversity_score` | KS-test 기반 통계적 구별 피처 수 |
| `run_advanced_eval` | 위 고급 지표 + 그래프 일괄 생성 |


In [ ]:
# ════════════════════════════════════════════════════════
# evaluation.py  — 기본 성능 평가 지표
# ════════════════════════════════════════════════════════

# ── 기본 지표 ────────────────────────────────────────────
def rmse_loss(ori_data, imputed_data, data_m):
    """결측 위치 RMSE (MinMax 정규화 후 계산)."""
    ori_norm, norm_params = normalization(ori_data)
    imp_norm, _           = normalization(imputed_data, norm_params)
    nom   = np.sum(((1 - data_m) * ori_norm - (1 - data_m) * imp_norm) ** 2)
    denom = np.sum(1 - data_m)
    return float(np.sqrt(nom / denom)) if denom > 0 else 0.0


def mae_loss(ori_data, imputed_data, data_m):
    """결측 위치 MAE (MinMax 정규화 후 계산)."""
    ori_norm, norm_params = normalization(ori_data)
    imp_norm, _           = normalization(imputed_data, norm_params)
    mask  = (1 - data_m)
    denom = np.sum(mask)
    return float(np.sum(np.abs(mask * ori_norm - mask * imp_norm)) / denom) if denom > 0 else 0.0


def auroc_score(ori_data, imputed_data, data_m):
    """결측 위치 실제값을 중앙값 기준 이진화 → AUROC."""
    ori_norm, norm_params = normalization(ori_data)
    imp_norm, _           = normalization(imputed_data, norm_params)
    mask = (1 - data_m).astype(bool)
    y_true_raw = ori_norm[mask]
    y_score    = imp_norm[mask]
    if len(y_true_raw) < 10:
        return float('nan')
    y_bin = (y_true_raw >= np.median(y_true_raw)).astype(int)
    if len(np.unique(y_bin)) < 2:
        return float('nan')
    try:
        return float(roc_auc_score(y_bin, y_score))
    except Exception:
        return float('nan')


def prediction_performance(ori_data, imputed_data):
    """마지막 컬럼을 레이블로 간주, 5-fold CV AUROC 측정."""
    try:
        X     = imputed_data[:, :-1]
        y     = ori_data[:, -1]
        unique = np.unique(y)
        if len(unique) < 2 or len(unique) > 20:
            return float('nan')

        # ✅ 핵심 수정: 반드시 0/1 정수로 강제 이진화
        median_val = np.nanmedian(y)
        y_bin = (y > median_val).astype(int)   # >= 대신 > 로 변경

        # ✅ 이진화 후에도 클래스 수 재확인
        if len(np.unique(y_bin)) < 2:
            return float('nan')

        # ✅ solver를 'lbfgs'로 변경 (multiclass도 처리 가능)
        clf    = LogisticRegression(
            solver='lbfgs',       # liblinear → lbfgs
            max_iter=500,
            random_state=42
        )
        scores = cross_val_score(clf, X, y_bin, cv=5, scoring='roc_auc')
        return float(np.mean(scores))
    except Exception as e:
        print(f'PredAUC Error: {e}')
        return float('nan')


def evaluate_all(ori_data, imputed_data, data_m):
    """기본 4개 지표 딕셔너리 반환."""
    return {
        'rmse'    : rmse_loss(ori_data, imputed_data, data_m),
        'mae'     : mae_loss(ori_data, imputed_data, data_m),
        'auroc'   : auroc_score(ori_data, imputed_data, data_m),
        'pred_auc': prediction_performance(ori_data, imputed_data),
    }


# ── 결측률별 테이블 ─────────────────────────────────────────
def table_by_miss_rate(results_by_rate, metric, model_names, miss_rates):
    """
    results_by_rate[miss_rate][model] = {rmse, mae, ...}
    → 결측률별 지표 테이블 문자열 반환
    """
    col_w  = 10
    header = f"  {'Model':28s}" + ''.join(f'{r*100:>{col_w}.0f}%' for r in miss_rates) + f"{'avg':>{col_w}s}"
    sep    = '─' * len(header)
    lines  = [sep, f'  {metric.upper()} by Missing Rate', sep, header, sep]
    for m in model_names:
        vals = [results_by_rate.get(r, {}).get(m, {}).get(metric, np.nan) for r in miss_rates]
        avg  = np.nanmean(vals)
        row  = f'  {m:28s}'
        row += ''.join(f'{v:>{col_w}.4f}' if not np.isnan(v) else f"{'N/A':>{col_w}s}" for v in vals)
        row += f'{avg:>{col_w}.4f}'
        lines.append(row)
    lines.append(sep)
    return '\n'.join(lines)


def table_db_model_rmse(results_by_ds_rate, model_names, datasets, miss_rates):
    """
    results_by_ds_rate[ds][miss_rate][model] = {rmse,...}
    → DB별 모델별 mean±std RMSE 테이블
    """
    lines  = ['─'*90, '  RMSE: mean ± std  (across missing rates)', '─'*90]
    header = f"  {'Model':28s}" + ''.join(f'{d:>14s}' for d in datasets)
    lines += [header, '─'*90]
    for m in model_names:
        row = f'  {m:28s}'
        for ds in datasets:
            vals = [results_by_ds_rate.get(ds, {}).get(r, {}).get(m, {}).get('rmse', np.nan)
                    for r in miss_rates]
            vals = [v for v in vals if not np.isnan(v)]
            row += f'{np.mean(vals):>7.4f}±{np.std(vals):.4f}' if vals else f"{'N/A':>14s}"
        lines.append(row)
    lines.append('─'*90)
    return '\n'.join(lines)


def plot_rmse_by_miss_rate(results_by_rate, model_names, miss_rates, title, save_path=None):
    """결측률별 RMSE 라인 그래프."""
    colors = ['steelblue', 'seagreen', 'darkorange', 'tomato', 'mediumpurple']
    fig, ax = plt.subplots(figsize=(8, 5))
    for model, color in zip(model_names, colors):
        vals = [results_by_rate.get(r, {}).get(model, {}).get('rmse', np.nan) for r in miss_rates]
        ax.plot([r*100 for r in miss_rates], vals, marker='o',
                label=model, color=color, linewidth=2)
    ax.set_xlabel('Missing Rate (%)'); ax.set_ylabel('RMSE ↓')
    ax.set_title(title); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path) or '.', exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()


print('✅ evaluation 함수 정의 완료')


In [ ]:
# ════════════════════════════════════════════════════════
# advanced_eval.py  — 고급 평가 지표
# ════════════════════════════════════════════════════════

# ── FLOPs & Parameters ──────────────────────────────────
def count_flops(model, input_dim=None):
    total = 0
    for m in model.modules():
        if isinstance(m, nn.Linear):
            total += 2 * m.in_features * m.out_features
    return total

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


# ── Latency ──────────────────────────────────────────────
def measure_latency(G, norm_data_x, data_m, device, runs=20):
    no, dim = norm_data_x.shape
    times   = []
    G.eval()
    for _ in range(runs):
        Z  = np.random.uniform(0, 0.01, (no, dim))
        X  = data_m * norm_data_x + (1 - data_m) * Z
        Xt = torch.tensor(X,      dtype=torch.float32).to(device)
        Mt = torch.tensor(data_m, dtype=torch.float32).to(device)
        t0 = time.perf_counter()
        with torch.no_grad():
            G(Xt, Mt)
        times.append((time.perf_counter() - t0) * 1000)
    return float(np.mean(times)), float(np.std(times))


# ── Gradient Magnitude 기록 (학습 내부 수집용, 시각화 제거) ──
def record_gradient_magnitudes(model):
    grad_norms = {}
    for name, param in model.named_parameters():
        if param.requires_grad and param.grad is not None and 'weight' in name:
            layer_id = name.split('.')[1] if '.' in name else 'out'
            grad_norms[f'Layer {layer_id}'] = param.grad.abs().mean().item()
    return grad_norms if grad_norms else {'none': 0.0}


# ── FID (간소화) ─────────────────────────────────────────
def compute_fid(real, fake):
    """FID ≈ ||μ_r - μ_g||² + Tr(Σ_r + Σ_g)"""
    mu_r, mu_g   = np.mean(real, axis=0), np.mean(fake, axis=0)
    cov_r, cov_g = np.cov(real.T), np.cov(fake.T)
    mean_diff    = float(np.sum((mu_r - mu_g) ** 2))
    if cov_r.ndim == 0:
        cov_trace = float(cov_r) + float(cov_g)
    else:
        cov_trace = np.trace(cov_r) + np.trace(cov_g)
    return float(mean_diff + cov_trace)


def plot_fid_convergence_single(fid_hist_gain, title='FID Convergence', save_path=None):
    """
    GAIN의 FID 수렴 곡선 단독 출력.
    (WGAN_FWAL은 학습 중 G를 반환하지 않으므로 FID history 없음)
    """
    if not fid_hist_gain:
        print(f'  [FID] history 없음 — 건너뜀: {title}')
        return
    fig, ax = plt.subplots(figsize=(8, 4))
    x = [i * CHECKPOINT_EVERY for i in range(1, len(fid_hist_gain) + 1)]
    ax.plot(x, fid_hist_gain, color='steelblue', linewidth=2, marker='o',
            markersize=3, label='GAIN')
    ax.set_xlabel('Iteration'); ax.set_ylabel('FID ↓')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path) or '.', exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()


def plot_fid_by_miss_rate(fid_hist_dict, miss_rates, title='FID Convergence by Miss Rate',
                           save_path=None):
    """
    결측률별 GAIN FID 수렴 곡선을 한 그래프에 겹쳐 비교.

    fid_hist_dict = {miss_rate: [fid_val, ...], ...}
    """
    cmap   = plt.cm.viridis
    colors = [cmap(i / max(len(miss_rates) - 1, 1)) for i in range(len(miss_rates))]

    fig, ax = plt.subplots(figsize=(9, 5))
    for (mr, hist), color in zip(fid_hist_dict.items(), colors):
        if not hist:
            continue
        x = [i * CHECKPOINT_EVERY for i in range(1, len(hist) + 1)]
        ax.plot(x, hist, linewidth=2, marker='o', markersize=3,
                label=f'miss={mr}', color=color)

    ax.set_xlabel('Iteration'); ax.set_ylabel('FID ↓')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(title='Missing Rate', fontsize=8); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path) or '.', exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()


def plot_fid_final_heatmap(fid_final_dict, title='Final FID by Dataset & Miss Rate',
                            save_path=None):
    """
    최종 FID 값을 데이터셋(행) × 결측률(열) 히트맵으로 비교.

    fid_final_dict = {ds: {miss_rate: fid_final_value, ...}, ...}
    """
    datasets   = list(fid_final_dict.keys())
    miss_rates = sorted({mr for d in fid_final_dict.values() for mr in d})
    matrix     = np.full((len(datasets), len(miss_rates)), np.nan)

    for i, ds in enumerate(datasets):
        for j, mr in enumerate(miss_rates):
            val = fid_final_dict[ds].get(mr)
            if val is not None:
                matrix[i, j] = val

    fig, ax = plt.subplots(figsize=(max(6, len(miss_rates) * 1.5),
                                    max(3, len(datasets) * 0.8)))
    im = ax.imshow(matrix, cmap='YlOrRd', aspect='auto')
    ax.set_xticks(range(len(miss_rates)))
    ax.set_xticklabels([f'{mr}' for mr in miss_rates])
    ax.set_yticks(range(len(datasets)))
    ax.set_yticklabels(datasets)
    ax.set_xlabel('Missing Rate'); ax.set_ylabel('Dataset')
    ax.set_title(title, fontsize=11, fontweight='bold')
    plt.colorbar(im, ax=ax, label='Final FID ↓')

    for i in range(len(datasets)):
        for j in range(len(miss_rates)):
            if not np.isnan(matrix[i, j]):
                ax.text(j, i, f'{matrix[i, j]:.3f}', ha='center', va='center',
                        fontsize=8, color='black')
    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path) or '.', exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()


# ── Diversity (KS-test 기반) ─────────────────────────────
def count_distinct_bins(real_missing, imputed_missing, alpha=0.05):
    if real_missing.ndim == 1:
        real_missing    = real_missing.reshape(-1, 1)
        imputed_missing = imputed_missing.reshape(-1, 1)
    distinct = 0
    for j in range(real_missing.shape[1]):
        r, g = real_missing[:, j], imputed_missing[:, j]
        if len(r) < 5 or np.std(r) < 1e-8:
            continue
        _, p = stats.ks_2samp(r, g)
        if p < alpha:
            distinct += 1
    return distinct

def diversity_score(ori_data, imputed_data, data_m):
    ori_norm, norm_params = normalization(ori_data)
    imp_norm, _           = normalization(imputed_data, norm_params)
    mask         = (1 - data_m).astype(bool)
    real_miss    = ori_norm[mask].reshape(-1, 1)
    imputed_miss = imp_norm[mask].reshape(-1, 1)
    return count_distinct_bins(real_miss, imputed_miss)


# ── 효율성 바 차트 ───────────────────────────────────────
def plot_efficiency_bars(metrics_dict, title='Efficiency Comparison', save_path=None):
    metric_keys   = ['params', 'flops', 'latency', 'diversity']
    metric_labels = ['Parameters', 'FLOPs', 'Latency (ms)', 'Distinct Bins ↓']
    model_names   = list(metrics_dict.keys())
    colors        = ['steelblue', 'seagreen', 'darkorange', 'tomato', 'mediumpurple']

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    fig.suptitle(title, fontsize=12, fontweight='bold')
    for ax, key, label in zip(axes, metric_keys, metric_labels):
        vals = [metrics_dict[m].get(key, 0) for m in model_names]
        bars = ax.bar(model_names, vals, color=colors[:len(model_names)], width=0.5)
        ax.set_title(label)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.02,
                    f'{v:.3g}', ha='center', va='bottom', fontsize=8)
        ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path) or '.', exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()


print('✅ advanced_eval 함수 정의 완료')

---
## 4. Light 데이터 로더 (`lightDBloader.py`)


In [ ]:
# ════════════════════════════════════════════════════════
# lightDBloader.py
# ════════════════════════════════════════════════════════

UCI_DATA_IDS = {
    'breast_cancer': 14,
    'spam'         : 94,
    'credit'       : 222,
    'wine'         : 109,
    'student'      : 697
}


def get_gain_data(dataset_name, missing_rate, seed):
    """
    UCI 저장소에서 데이터를 받아 GAIN 실험용 numpy 배열을 반환합니다.

    Returns
    -------
    ori_data_x  : 원본 데이터 (결측 없음)
    miss_data_x : MCAR 결측치 주입 데이터 (NaN)
    data_m      : 마스크 행렬 (1=관측, 0=결측)
    """
    dataset_id = UCI_DATA_IDS[dataset_name]
    dataset    = fetch_ucirepo(id=dataset_id)
    X_df       = dataset.data.features.copy()

    # 자연 결측치 채우기
    for col in X_df.columns:
        if X_df[col].isnull().sum() > 0:
            if not pd.api.types.is_numeric_dtype(X_df[col]):
                X_df[col] = X_df[col].fillna(X_df[col].mode()[0])
            else:
                X_df[col] = X_df[col].fillna(X_df[col].mean())

    X_df       = pd.get_dummies(X_df, drop_first=True)
    X_original = X_df.values.astype(float)

    # MCAR 결측치 생성
    np.random.seed(seed)
    mask        = (np.random.rand(*X_original.shape) > missing_rate).astype(float)
    X_corrupted = X_original.copy()
    X_corrupted[mask == 0] = np.nan

    return X_original, X_corrupted, mask


def plot_missing_heatmap(mask, dataset_name, missing_rate, save_path=None):
    """결측 패턴 히트맵 시각화."""
    plt.figure(figsize=(10, 6))
    sns.heatmap(mask, cmap='binary', cbar=False)
    plt.title(f"Missing Pattern ({dataset_name}, Rate={missing_rate*100:.0f}%)", fontsize=14)
    plt.xlabel("Features"); plt.ylabel("Instances")
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


def plot_data_distribution(X_orig, X_corr, feature_idx=0, save_path=None):
    """특정 피처에 대해 원본 vs 결측 처리 데이터 분포 비교."""
    plt.figure(figsize=(9, 4))
    corr_vals = X_corr[:, feature_idx]
    corr_vals = corr_vals[~np.isnan(corr_vals)]
    sns.kdeplot(X_orig[:, feature_idx], label='Original',  fill=True, color='steelblue', alpha=0.6)
    sns.kdeplot(corr_vals,              label='Corrupted', fill=True, color='tomato',    alpha=0.4, linestyle='--')
    plt.title(f"Distribution Comparison (Feature {feature_idx})")
    plt.legend()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


print("✅ lightDBloader 함수 정의 완료")

---
## 5. Heavy 데이터 로더 (`heavyDBloader.py`)

> 파일 경로 (노트북 기준 상위 디렉토리):  
> - HIGGS : `../data/HIGGS.csv`  
> - Criteo: `../criteoDB/train.txt`, `../criteoDB/test.txt`, `../criteoDB/readme.txt`


In [ ]:
# ════════════════════════════════════════════════════════
# heavyDBloader.py
# ════════════════════════════════════════════════════════
import os
from sklearn.preprocessing import StandardScaler

# ── 파일 경로 (노트북 파일 기준 상위 폴더) ──────────────────────────
HIGGS_PATH   = os.path.join('.', 'data', 'HIGGS.csv')
CRITEO_TRAIN = os.path.join('.', 'data', 'criteoDB', 'train.txt')
CRITEO_TEST  = os.path.join('.', 'data', 'criteoDB', 'test.txt')
CRITEO_README= os.path.join('.', 'data', 'criteoDB', 'readme.txt')
# ────────────────────────────────────────────────────────


class HeavyImputationDataset(Dataset):
    """
    HIGGS / Criteo 대용량 데이터셋 로더.

    Parameters
    ----------
    file_path    : CSV(HIGGS) 또는 TSV(Criteo) 파일 경로
    missing_rate : MCAR 결측률
    seed         : 재현성 시드
    nrows        : 메모리 절약을 위해 읽을 최대 행 수 (기본 500,000)
    dataset_type : 'higgs' 또는 'criteo'
    """
    def __init__(self, file_path, missing_rate, seed,
                 nrows=500_000, dataset_type='higgs'):
        print(f"[{dataset_type.upper()}] Loading {nrows:,} rows from {file_path} ...")

        if not os.path.exists(file_path):
            raise FileNotFoundError(
                f"파일을 찾을 수 없습니다: {os.path.abspath(file_path)}\n"
                f"  HIGGS  → {os.path.abspath(HIGGS_PATH)}\n"
                f"  Criteo → {os.path.abspath(CRITEO_TRAIN)}"
            )

        sep    = '\t' if dataset_type == 'criteo' else ','
        header = None if dataset_type == 'criteo' else 'infer'

        df = pd.read_csv(file_path, sep=sep, nrows=nrows, header=header)
        # [버그수정] Criteo: col1~13만 수치형, col14~39는 hex → astype(float32) ValueError
        # 알고리즘 변경 없이 데이터 로드 오류만 수정
        if dataset_type == 'criteo':
            y_raw  = df.iloc[:, 0].values
            df_num = df.iloc[:, 1:14].apply(pd.to_numeric, errors='coerce')
            X_raw  = np.nan_to_num(df_num.values.astype(np.float32))
        else:
            y_raw = df.iloc[:, 0].values
            X_raw = np.nan_to_num(df.iloc[:, 1:].values.astype(np.float32))

        scaler     = StandardScaler()
        X_original = scaler.fit_transform(X_raw)          # float64

        self.X_orig = torch.from_numpy(X_original.astype(np.float32))
        self.y      = torch.from_numpy(y_raw.astype(np.int64))

        # MCAR 마스크 생성 (1=관측, 0=결측)
        np.random.seed(seed)
        mask_np  = (np.random.rand(*X_original.shape) > missing_rate).astype(np.float32)
        self.mask = torch.from_numpy(mask_np)

        # 결측 위치를 0으로 채운 corrupted 버전
        self.X_corr = self.X_orig.clone()
        self.X_corr[self.mask == 0] = 0.0

        print(f"[{dataset_type.upper()}] 로드 완료. Shape: {tuple(self.X_orig.shape)}")

    def __len__(self):
        return len(self.X_orig)

    def __getitem__(self, idx):
        return self.X_orig[idx], self.X_corr[idx], self.mask[idx], self.y[idx]


def get_heavy_dataloader(dataset_name, missing_rate, seed,
                         batch_size=512, nrows=500_000, split='train'):
    """
    HIGGS / Criteo DataLoader 반환.

    Parameters
    ----------
    dataset_name : 'higgs' 또는 'criteo'
    missing_rate : MCAR 결측률
    seed         : 재현성 시드
    batch_size   : 미니배치 크기 (기본 512)
    nrows        : 읽을 최대 행 수 (기본 500,000)
    split        : 'train' 또는 'test'  (Criteo 전용, HIGGS는 단일 파일)

    Returns
    -------
    loader : torch DataLoader
    """
    if dataset_name.lower() == 'higgs':
        file_path = HIGGS_PATH
    elif dataset_name.lower() == 'criteo':
        file_path = CRITEO_TRAIN if split == 'train' else CRITEO_TEST
    else:
        raise ValueError("dataset_name 은 'higgs' 또는 'criteo' 이어야 합니다.")

    dataset = HeavyImputationDataset(
        file_path=file_path,
        missing_rate=missing_rate,
        seed=seed,
        nrows=nrows,
        dataset_type=dataset_name.lower()
    )
    loader = DataLoader(dataset, batch_size=batch_size,
                        shuffle=(split == 'train'), num_workers=0,
                        pin_memory=torch.cuda.is_available())
    return loader


def plot_heavy_missing_heatmap(mask, dataset_name, missing_rate,
                               sample_size=1000, save_path=None):
    """대용량 데이터 결측 패턴 히트맵 (앞 sample_size 행만 표시)."""
    if isinstance(mask, torch.Tensor):
        mask = mask.numpy()
    plot_data = mask[:sample_size, :]

    plt.figure(figsize=(12, 8))
    sns.heatmap(plot_data, cmap='binary', cbar=False)
    plt.title(f"Missing Pattern Sample ({dataset_name}, "
              f"Rate={missing_rate*100:.0f}%, N={sample_size})", fontsize=14)
    plt.xlabel('Feature Index'); plt.ylabel('Sample Index')
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


def plot_heavy_distribution(X_orig, X_corr, feature_idx=0,
                            sample_size=5000, save_path=None):
    """대용량 데이터의 특정 피처에 대해 원본 vs 결측 처리 분포 비교."""
    if isinstance(X_orig, torch.Tensor): X_orig = X_orig.numpy()
    if isinstance(X_corr, torch.Tensor): X_corr = X_corr.numpy()

    indices  = np.random.choice(len(X_orig), min(sample_size, len(X_orig)), replace=False)
    s_orig   = X_orig[indices, feature_idx]
    s_corr   = X_corr[indices, feature_idx]
    s_corr   = s_corr[s_corr != 0]          # 결측(0으로 채운) 위치 제외

    plt.figure(figsize=(9, 4))
    sns.kdeplot(s_orig,  label='Original',  fill=True, color='steelblue', alpha=0.6)
    sns.kdeplot(s_corr,  label='Corrupted', fill=True, color='tomato',    alpha=0.4,
                linestyle='--')
    plt.title(f"Feature Distribution — Feature {feature_idx} (N={sample_size})")
    plt.xlabel('Standardized Value'); plt.ylabel('Density')
    plt.legend()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


print("✅ heavyDBloader 함수 정의 완료")
print(f"  HIGGS   경로 확인: {os.path.abspath(HIGGS_PATH)}")
print(f"  Criteo  경로 확인: {os.path.abspath(CRITEO_TRAIN)}")


### (선택) Heavy 데이터 로더 사용 예시

> 아래 셀은 실제 HIGGS/Criteo 파일이 있을 때만 실행하세요.  
> 파일이 없으면 건너뛰어도 이후 Light 실험에 영향 없습니다.

In [ ]:
# ── Heavy 데이터셋 사용 예시 (파일 존재 시에만 실행) ──
HEAVY_DATASET = 'higgs'   # 또는 'criteo'
HEAVY_MISS    = 0.2
HEAVY_NROWS   = 200_000   # RAM에 맞게 조정

loader = get_heavy_dataloader(
    dataset_name = HEAVY_DATASET,
    missing_rate = HEAVY_MISS,
    seed         = 42,
    batch_size   = 512,
    nrows        = HEAVY_NROWS
)

# 첫 배치 확인
X_orig_b, X_corr_b, mask_b, y_b = next(iter(loader))
print('X_orig shape :', X_orig_b.shape)
print('mask  shape  :', mask_b.shape)

# 결측 패턴 시각화
plot_heavy_missing_heatmap(mask_b, HEAVY_DATASET, HEAVY_MISS, sample_size=256)
plot_heavy_distribution(X_orig_b, X_corr_b, feature_idx=0)
# print("Heavy 데이터셋 사용 시 위 주석을 해제하세요.")

---
## 6. 모델 정의 — GAIN (`model_GAIN.py`)


In [ ]:
# ════════════════════════════════════════════════════════
# model_GAIN.py
# ════════════════════════════════════════════════════════

class GAINGenerator(nn.Module):
    """GAIN Generator: (X, M) → imputed values."""
    def __init__(self, dim, h_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim * 2, h_dim), nn.ReLU(),
            nn.Linear(h_dim, h_dim),   nn.ReLU(),
            nn.Linear(h_dim, dim),     nn.Sigmoid()
        )
        self._init_weights()

    def _init_weights(self):
        for layer in self.net:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_normal_(layer.weight)
                nn.init.zeros_(layer.bias)

    def forward(self, x, m):
        return self.net(torch.cat([x, m], dim=1))


class GAINDiscriminator(nn.Module):
    """GAIN Discriminator: (X_hat, H) → observed/imputed probability."""
    def __init__(self, dim, h_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim * 2, h_dim), nn.ReLU(),
            nn.Linear(h_dim, h_dim),   nn.ReLU(),
            nn.Linear(h_dim, dim),     nn.Sigmoid()
        )
        self._init_weights()

    def _init_weights(self):
        for layer in self.net:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_normal_(layer.weight)
                nn.init.zeros_(layer.bias)

    def forward(self, x, h):
        return self.net(torch.cat([x, h], dim=1))


def gain(data_x, gain_parameters):
    """
    GAIN 기반 결측치 대체 (PyTorch).

    Parameters
    ----------
    data_x          : 결측치가 포함된 numpy 배열
    gain_parameters : dict {batch_size, hint_rate, alpha, iterations, device}

    Returns
    -------
    imputed_data : 대체 완료된 numpy 배열
    """
    device = torch.device(gain_parameters.get('device', 'cpu'))
    data_m = 1 - np.isnan(data_x)

    batch_size = gain_parameters['batch_size']
    hint_rate  = gain_parameters['hint_rate']
    alpha      = gain_parameters['alpha']
    iterations = gain_parameters['iterations']

    no, dim = data_x.shape
    h_dim   = dim

    norm_data, norm_parameters = normalization(data_x)
    norm_data_x = np.nan_to_num(norm_data, nan=0.0)

    generator     = GAINGenerator(dim, h_dim).to(device)
    discriminator = GAINDiscriminator(dim, h_dim).to(device)
    D_optimizer   = optim.Adam(discriminator.parameters())
    G_optimizer   = optim.Adam(generator.parameters())

    use_amp = gain_parameters.get('use_amp', USE_AMP)
    scaler  = GradScaler(enabled=use_amp)

    for it in tqdm(range(iterations), desc="[GAIN] Training", leave=False):
        batch_idx  = sample_batch_index(no, batch_size)
        X_mb       = norm_data_x[batch_idx, :]
        M_mb       = data_m[batch_idx, :]
        Z_mb       = uniform_sampler(0, 0.01, batch_size, dim)
        H_mb       = M_mb * binary_sampler(hint_rate, batch_size, dim)
        X_mb       = M_mb * X_mb + (1 - M_mb) * Z_mb

        X_t = torch.tensor(X_mb, dtype=torch.float32).to(device)
        M_t = torch.tensor(M_mb, dtype=torch.float32).to(device)
        H_t = torch.tensor(H_mb, dtype=torch.float32).to(device)

        # Discriminator step — forward in float16, loss in float32
        discriminator.train(); generator.eval()
        with torch.no_grad():
            with autocast(enabled=use_amp):
                G_sample = generator(X_t, M_t)
        Hat_X = X_t * M_t + G_sample * (1 - M_t)
        with autocast(enabled=use_amp):
            D_prob = discriminator(Hat_X.detach(), H_t)
        D_prob_f32 = D_prob.float()
        D_loss = -torch.mean(
            M_t * torch.log(D_prob_f32 + 1e-8) +
            (1 - M_t) * torch.log(1.0 - D_prob_f32 + 1e-8)
        )
        D_optimizer.zero_grad()
        scaler.scale(D_loss).backward()
        scaler.step(D_optimizer)
        scaler.update()

        # Generator step
        generator.train(); discriminator.eval()
        with autocast(enabled=use_amp):
            G_sample   = generator(X_t, M_t)
            Hat_X      = X_t * M_t + G_sample * (1 - M_t)
            D_prob     = discriminator(Hat_X, H_t)
        D_prob_f32  = D_prob.float()
        G_sample_f32= G_sample.float()
        G_loss_adv  = -torch.mean((1 - M_t) * torch.log(D_prob_f32 + 1e-8))
        MSE_loss    = torch.mean((M_t * X_t - M_t * G_sample_f32) ** 2) / (torch.mean(M_t) + 1e-8)
        G_loss      = G_loss_adv + alpha * MSE_loss
        G_optimizer.zero_grad()
        scaler.scale(G_loss).backward()
        scaler.step(G_optimizer)
        scaler.update()

    # Inference
    generator.eval()
    Z_inf = uniform_sampler(0, 0.01, no, dim)
    X_inf = data_m * norm_data_x + (1 - data_m) * Z_inf
    with torch.no_grad():
        imputed = generator(
            torch.tensor(X_inf, dtype=torch.float32).to(device),
            torch.tensor(data_m, dtype=torch.float32).to(device)
        ).cpu().numpy()

    imputed = data_m * norm_data_x + (1 - data_m) * imputed
    imputed = renormalization(imputed, norm_parameters)
    imputed = rounding(imputed, data_x)
    return imputed


print("✅ GAIN 모델 정의 완료")

---
## 7. 모델 정의 — WGAN-FWAL (`model_WGANwithFWAL.py`)


In [ ]:
# ════════════════════════════════════════════════════════
# model_WGANwithFWAL.py
# ════════════════════════════════════════════════════════

class FWALGenerator(nn.Module):
    def __init__(self, dim, h_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim * 2, h_dim), nn.ReLU(),
            nn.Linear(h_dim, h_dim),   nn.ReLU(),
            nn.Linear(h_dim, dim),     nn.Sigmoid()
        )
        self._init_weights()

    def _init_weights(self):
        for l in self.net:
            if isinstance(l, nn.Linear):
                nn.init.xavier_normal_(l.weight); nn.init.zeros_(l.bias)

    def forward(self, x, m):
        return self.net(torch.cat([x, m], dim=1))


from torch.nn.utils import spectral_norm

class FWALCritic(nn.Module):
    def __init__(self, dim, h_dim):
        super().__init__()
        self.net = nn.Sequential(
            spectral_norm(nn.Linear(dim * 2, h_dim)), nn.ReLU(),
            spectral_norm(nn.Linear(h_dim, h_dim)),   nn.ReLU(),
            spectral_norm(nn.Linear(h_dim, dim))
        )
        self._init_weights()

    def _init_weights(self):
        for l in self.net:
            if isinstance(l, nn.Linear):
                nn.init.xavier_normal_(l.weight_orig)
                nn.init.zeros_(l.bias)

    def forward(self, x, h):
        return self.net(torch.cat([x, h], dim=1))


def _gradient_penalty_fwal(critic, real, fake, h, device):
    alpha       = torch.rand(real.size(0), 1).to(device)
    interpolate = (alpha * real + (1 - alpha) * fake).requires_grad_(True)
    d_interp    = critic(interpolate, h)
    grads       = torch.autograd.grad(
        outputs=d_interp, inputs=interpolate,
        grad_outputs=torch.ones_like(d_interp),
        create_graph=True, retain_graph=True, only_inputs=True
    )[0]
    grads = grads.view(grads.size(0), -1)
    return ((grads.norm(2, dim=1) - 1) ** 2).mean()


def wganwithFWAL(data_x, gain_parameters):
    """WGAN-GP + FWAL 결측치 대체. Returns (imputed_numpy, trained_generator)."""
    device     = torch.device(gain_parameters.get('device', 'cpu'))
    data_m     = 1 - np.isnan(data_x)
    batch_size = gain_parameters['batch_size']
    hint_rate  = gain_parameters['hint_rate']
    alpha_gain = gain_parameters['alpha']
    iterations = gain_parameters['iterations']
    _lr        = gain_parameters.get('lr', 1e-4)
    use_coslr  = gain_parameters.get('use_coslr', True)
    lambda_gp  = 10.0
    n_critic   = 5

    no, dim = data_x.shape
    h_dim   = dim

    norm_data, norm_parameters = normalization(data_x)
    norm_data_x = np.nan_to_num(norm_data, nan=0.0)

    generator = FWALGenerator(dim, h_dim).to(device)
    critic    = FWALCritic(dim, h_dim).to(device)
    G_opt     = optim.Adam(generator.parameters(), lr=_lr, betas=(0.5, 0.9))
    C_opt     = optim.Adam(critic.parameters(),    lr=_lr, betas=(0.5, 0.9))

    if use_coslr:
        G_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
            G_opt, T_max=iterations, eta_min=_lr * 0.01)
        C_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
            C_opt, T_max=iterations, eta_min=_lr * 0.01)

    use_amp = gain_parameters.get('use_amp', USE_AMP)
    scaler  = GradScaler(enabled=use_amp)
    g_grad_hist = []

    for it in tqdm(range(iterations), desc="[WGAN-FWAL] Training", leave=False):
        # ── Critic steps ──────────────────────────────────────
        for _ in range(n_critic):
            batch_idx  = sample_batch_index(no, batch_size)
            X_mb       = norm_data_x[batch_idx, :]
            M_mb       = data_m[batch_idx, :]
            Z_mb       = uniform_sampler(0, 0.01, batch_size, dim)
            H_mb       = M_mb * binary_sampler(hint_rate, batch_size, dim)
            X_mb_noisy = M_mb * X_mb + (1 - M_mb) * Z_mb

            X_t = torch.tensor(X_mb_noisy, dtype=torch.float32).to(device)
            M_t = torch.tensor(M_mb,       dtype=torch.float32).to(device)
            H_t = torch.tensor(H_mb,       dtype=torch.float32).to(device)

            # Critic forward in autocast (float16 행렬곱)
            with autocast(enabled=use_amp):
                with torch.no_grad():
                    G_sample = generator(X_t, M_t)
                Hat_X   = X_t * M_t + G_sample * (1 - M_t)
                C_score = critic(Hat_X.detach(), H_t)
                C_adv   = (torch.mean((1 - M_t) * C_score)
                           - torch.mean(M_t * C_score)).float()
            # Gradient penalty는 반드시 float32 (autograd.grad 수치 안정성)
            gp     = _gradient_penalty_fwal(critic, X_t, Hat_X.detach().float(), H_t, device)
            C_loss = C_adv + lambda_gp * gp
            C_opt.zero_grad()
            scaler.scale(C_loss).backward()
            scaler.step(C_opt)
            scaler.update()

        # ── Generator step: 독립 배치 (Critic 마지막 배치와 분리) ──
        g_idx     = sample_batch_index(no, batch_size)
        X_g       = norm_data_x[g_idx, :]
        M_g       = data_m[g_idx, :]
        Z_g       = uniform_sampler(0, 0.01, batch_size, dim)
        H_g       = M_g * binary_sampler(hint_rate, batch_size, dim)
        X_g_noisy = M_g * X_g + (1 - M_g) * Z_g
        Xg_t = torch.tensor(X_g_noisy, dtype=torch.float32).to(device)
        Mg_t = torch.tensor(M_g,       dtype=torch.float32).to(device)
        Hg_t = torch.tensor(H_g,       dtype=torch.float32).to(device)

        with autocast(enabled=use_amp):
            G_sample  = generator(Xg_t, Mg_t)
            Hat_Xg    = Xg_t * Mg_t + G_sample * (1 - Mg_t)
            C_score_g = critic(Hat_Xg, Hg_t)
        G_loss_adv = (-torch.mean((1 - Mg_t) * C_score_g)).float()
        MSE_loss   = (torch.mean((Mg_t * Xg_t - Mg_t * G_sample.float()) ** 2)
                      / (torch.mean(Mg_t) + 1e-8))
        G_loss = G_loss_adv + alpha_gain * MSE_loss
        G_opt.zero_grad()
        scaler.scale(G_loss).backward()
        scaler.unscale_(G_opt)   # unscale 먼저 해야 clip_grad_norm 정확
        torch.nn.utils.clip_grad_norm_(generator.parameters(), max_norm=1.0)
        scaler.step(G_opt)
        scaler.update()

        if use_coslr:
            G_sched.step()
            C_sched.step()
        if (it + 1) % 100 == 0:
            g_grad_hist.append(record_gradient_magnitudes(generator))

    generator.eval()
    with torch.no_grad():
        imputed = generator(
            torch.tensor(norm_data_x, dtype=torch.float32).to(device),
            torch.tensor(data_m,      dtype=torch.float32).to(device)
        ).cpu().numpy()

    imputed = data_m * norm_data_x + (1 - data_m) * imputed
    imputed = renormalization(imputed, norm_parameters)
    imputed = rounding(imputed, data_x)
    generator._grad_hist = g_grad_hist
    return imputed, generator


print("\u2705 WGAN-FWAL 모델 정의 완료")


---
## 8. Knowledge Distillation — 2단계 (`model_KD.py`)

> **1단계 (Hard-label)**: Teacher(WGAN-FWAL) imputed output → Student MSE  
> **2단계 (Soft+Feature)**: Teacher 중간 feature L2 + KL Divergence  


In [ ]:
# ════════════════════════════════════════════════════════
# model_KD.py  —  2단계 Knowledge Distillation
# ════════════════════════════════════════════════════════
import torch.nn.functional as F


# ── Student Generator (경량 버전, hidden dim = dim//2) ───
class StudentGenerator(nn.Module):
    def __init__(self, dim, h_dim=None):
        super().__init__()
        h_dim = h_dim or max(dim // 2, 16)
        self.fc1 = nn.Linear(dim * 2, h_dim)
        self.fc2 = nn.Linear(h_dim, h_dim)
        self.fc3 = nn.Linear(h_dim, dim)
        self._init_weights()

    def _init_weights(self):
        for layer in [self.fc1, self.fc2, self.fc3]:
            nn.init.xavier_normal_(layer.weight)
            nn.init.zeros_(layer.bias)

    def forward(self, x, m, return_feat=False):
        inp  = torch.cat([x, m], dim=1)
        h1   = F.relu(self.fc1(inp))
        feat = F.relu(self.fc2(h1))
        out  = torch.sigmoid(self.fc3(feat))
        if return_feat:
            return out, feat
        return out


class _FeatureHook:
    def __init__(self):
        self.feat = None
    def hook_fn(self, module, inp, output):
        self.feat = output


# ── 1단계: Hard-label Distillation ────────────────────
def kd_stage1(teacher_gen, student_gen, norm_data_x, data_m, device, params):
    batch_size = params['batch_size']
    alpha      = params.get('kd_alpha', 1.0)
    iters      = params.get('kd_stage1_iters', 3000)
    no, dim    = norm_data_x.shape

    use_amp = params.get('use_amp', USE_AMP)
    scaler  = GradScaler(enabled=use_amp)
    optimizer = optim.Adam(student_gen.parameters(), lr=1e-3)
    teacher_gen.eval(); student_gen.train()

    for it in tqdm(range(iters), desc="[KD Stage-1] Hard-label", leave=False):
        idx  = sample_batch_index(no, batch_size)
        X_mb = norm_data_x[idx, :]
        M_mb = data_m[idx, :]
        Z_mb = uniform_sampler(0, 0.01, batch_size, dim)
        X_in = M_mb * X_mb + (1 - M_mb) * Z_mb

        X_t  = torch.tensor(X_in,  dtype=torch.float32).to(device)
        M_t  = torch.tensor(M_mb,  dtype=torch.float32).to(device)
        X_obs= torch.tensor(X_mb,  dtype=torch.float32).to(device)

        # Teacher + Student forward in autocast
        with autocast(enabled=use_amp):
            with torch.no_grad():
                t_out = teacher_gen(X_t, M_t)
            s_out_amp = student_gen(X_t, M_t)
        t_out_f32 = t_out.float()
        s_out     = s_out_amp.float()

        missing_count = (1 - M_t).sum() + 1e-8
        loss_pseudo   = ((1 - M_t) * (s_out - t_out_f32) ** 2).sum() / missing_count
        loss_recon    = torch.mean(M_t * (s_out - X_obs) ** 2) / (M_t.mean() + 1e-8)
        loss          = loss_pseudo + alpha * loss_recon
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

    return student_gen


# ── 2단계: Soft-label + Feature Distillation (논문 수식) ─
def kd_stage2(teacher_gen, student_gen, norm_data_x, data_m, device, params):
    """
    Loss = w_feat * L2(s_feat, t_feat)
         + w_kl  * T^2 * KL(log_softmax(s/T) || softmax(t/T))  <- 논문 수식
         + alpha * MSE(s_out, observed)
    """
    batch_size = params['batch_size']
    alpha      = params.get('kd_alpha', 1.0)
    iters      = params.get('kd_stage2_iters', 3000)
    T          = params.get('kd_temperature', 3.0)
    w_feat     = params.get('kd_w_feat', 0.5)
    w_kl       = params.get('kd_w_kl',  0.5)
    no, dim    = norm_data_x.shape

    hook   = _FeatureHook()
    handle = teacher_gen.net[3].register_forward_hook(hook.hook_fn)

    use_amp = params.get('use_amp', USE_AMP)
    scaler  = GradScaler(enabled=use_amp)
    optimizer = optim.Adam(student_gen.parameters(), lr=5e-4)
    teacher_gen.eval(); student_gen.train()

    for it in tqdm(range(iters), desc="[KD Stage-2] Soft+Feature", leave=False):
        idx  = sample_batch_index(no, batch_size)
        X_mb = norm_data_x[idx, :]
        M_mb = data_m[idx, :]
        Z_mb = uniform_sampler(0, 0.01, batch_size, dim)
        X_in = M_mb * X_mb + (1 - M_mb) * Z_mb

        X_t  = torch.tensor(X_in,  dtype=torch.float32).to(device)
        M_t  = torch.tensor(M_mb,  dtype=torch.float32).to(device)
        X_obs= torch.tensor(X_mb,  dtype=torch.float32).to(device)

        # Forward in autocast
        with autocast(enabled=use_amp):
            with torch.no_grad():
                t_out  = teacher_gen(X_t, M_t)
            t_feat_amp = hook.feat.detach()
            s_out_amp, s_feat_amp = student_gen(X_t, M_t, return_feat=True)
        # Loss in float32 (KL / MSE 수치 안정성)
        t_out      = t_out.float()
        t_feat     = t_feat_amp.float()
        s_out      = s_out_amp.float()
        s_feat     = s_feat_amp.float()

        if s_feat.shape[1] != t_feat.shape[1]:
            if not hasattr(kd_stage2, '_proj'):
                kd_stage2._proj = nn.Linear(s_feat.shape[1], t_feat.shape[1], bias=False).to(device)
            s_feat_proj = kd_stage2._proj(s_feat)
        else:
            s_feat_proj = s_feat
        loss_feat = F.mse_loss(s_feat_proj, t_feat)

        # KL Divergence — 논문 수식: T^2 * KL(log_softmax(s/T) || softmax(t/T))
        s_soft  = F.log_softmax(s_out / T, dim=1)
        t_soft  = F.softmax(t_out / T, dim=1)
        loss_kl = F.kl_div(s_soft, t_soft, reduction='batchmean') * (T ** 2)

        loss_recon = torch.mean(M_t * (s_out - X_obs) ** 2) / (M_t.mean() + 1e-8)
        loss = w_feat * loss_feat + w_kl * loss_kl + alpha * loss_recon
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

    handle.remove()
    if hasattr(kd_stage2, '_proj'):
        del kd_stage2._proj
    return student_gen


# ── 통합 KD 파이프라인 ────────────────────────────────
def knowledge_distillation(data_x, gain_parameters):
    """
    2단계 KD.
    Teacher = wganwithFWAL 로 학습된 실제 FWALGenerator (가짜 proxy 아님).
    Returns (imputed_data, student_generator).
    """
    device     = torch.device(gain_parameters.get('device', 'cpu'))
    data_m     = 1 - np.isnan(data_x)
    iterations = gain_parameters['iterations']
    gain_parameters.setdefault('kd_stage1_iters', iterations // 2)
    gain_parameters.setdefault('kd_stage2_iters', iterations // 2)

    no, dim = data_x.shape
    norm_data, norm_parameters = normalization(data_x)
    norm_data_x = np.nan_to_num(norm_data, nan=0.0)

    # Step 0: 실제 FWALGenerator를 Teacher로 사용 (가짜 proxy 제거)
    print("[KD] Teacher(WGAN-FWAL) 학습 중 ...")
    _, teacher_gen = wganwithFWAL(data_x, gain_parameters)
    teacher_gen.eval()
    print("[KD] Teacher 학습 완료.")

    # Step 1: Hard-label
    print("[KD] Stage 1 — Hard-label Distillation ...")
    student_gen = StudentGenerator(dim).to(device)
    student_gen = kd_stage1(teacher_gen, student_gen,
                             norm_data_x, data_m, device, gain_parameters)
    print("[KD] Stage 1 완료.")

    # Step 2: Soft-label + Feature
    print("[KD] Stage 2 — Soft-label + Feature Distillation ...")
    student_gen = kd_stage2(teacher_gen, student_gen,
                             norm_data_x, data_m, device, gain_parameters)
    print("[KD] Stage 2 완료.")

    student_gen.eval()
    Z_inf = uniform_sampler(0, 0.01, no, dim)
    X_inf = data_m * norm_data_x + (1 - data_m) * Z_inf
    with torch.no_grad():
        imputed = student_gen(
            torch.tensor(X_inf,  dtype=torch.float32).to(device),
            torch.tensor(data_m, dtype=torch.float32).to(device)
        ).cpu().numpy()

    imputed = data_m * norm_data_x + (1 - data_m) * imputed
    imputed = renormalization(imputed, norm_parameters)
    imputed = rounding(imputed, data_x)
    return imputed, student_gen


print("\u2705 Knowledge Distillation (2단계) 모델 정의 완료")


In [ ]:
# ── Student Baseline (KD 없이 직접 학습) ───────────────────
def student_baseline(data_x, gain_parameters):
    """
    KD 없이 StudentGenerator를 GAIN 방식으로 직접 학습.
    Teacher 없이 Student 구조만으로 학습했을 때의 성능 baseline.

    Loss = G_adv (BCE) + alpha * L_rec
    """
    device     = torch.device(gain_parameters.get('device', 'cpu'))
    data_m     = 1 - np.isnan(data_x)
    batch_size = gain_parameters['batch_size']
    hint_rate  = gain_parameters['hint_rate']
    alpha      = gain_parameters['alpha']
    iterations = gain_parameters['iterations']

    no, dim = data_x.shape

    norm_data, norm_parameters = normalization(data_x)
    norm_data_x = np.nan_to_num(norm_data, nan=0.0)

    # Student 크기 Generator + 풀사이즈 Discriminator
    student_gen  = StudentGenerator(dim).to(device)
    discriminator = GAINDiscriminator(dim, dim).to(device)
    D_opt = optim.Adam(discriminator.parameters())
    G_opt = optim.Adam(student_gen.parameters())

    use_amp = gain_parameters.get('use_amp', USE_AMP)
    scaler  = GradScaler(enabled=use_amp)

    for it in tqdm(range(iterations), desc="[Student-Baseline] Training", leave=False):
        idx  = sample_batch_index(no, batch_size)
        X_mb = norm_data_x[idx, :]
        M_mb = data_m[idx, :]
        Z_mb = uniform_sampler(0, 0.01, batch_size, dim)
        H_mb = M_mb * binary_sampler(hint_rate, batch_size, dim)
        X_mb = M_mb * X_mb + (1 - M_mb) * Z_mb

        X_t = torch.tensor(X_mb, dtype=torch.float32).to(device)
        M_t = torch.tensor(M_mb, dtype=torch.float32).to(device)
        H_t = torch.tensor(H_mb, dtype=torch.float32).to(device)

        # Discriminator step
        discriminator.train(); student_gen.eval()
        with torch.no_grad():
            with autocast(enabled=use_amp):
                G_sample = student_gen(X_t, M_t)
        Hat_X = X_t * M_t + G_sample * (1 - M_t)
        with autocast(enabled=use_amp):
            D_prob = discriminator(Hat_X.detach(), H_t)
        D_prob_f32 = D_prob.float()
        D_loss = -torch.mean(
            M_t * torch.log(D_prob_f32 + 1e-8) +
            (1 - M_t) * torch.log(1.0 - D_prob_f32 + 1e-8)
        )
        D_opt.zero_grad()
        scaler.scale(D_loss).backward()
        scaler.step(D_opt)
        scaler.update()

        # Generator step
        student_gen.train(); discriminator.eval()
        with autocast(enabled=use_amp):
            G_sample   = student_gen(X_t, M_t)
            Hat_X      = X_t * M_t + G_sample * (1 - M_t)
            D_prob     = discriminator(Hat_X, H_t)
        D_prob_f32  = D_prob.float()
        G_sample_f32= G_sample.float()
        G_loss_adv  = -torch.mean((1 - M_t) * torch.log(D_prob_f32 + 1e-8))
        MSE_loss    = torch.mean((M_t * X_t - M_t * G_sample_f32) ** 2) / (torch.mean(M_t) + 1e-8)
        G_loss      = G_loss_adv + alpha * MSE_loss
        G_opt.zero_grad()
        scaler.scale(G_loss).backward()
        scaler.step(G_opt)
        scaler.update()

    # Inference
    student_gen.eval()
    Z_inf = uniform_sampler(0, 0.01, no, dim)
    X_inf = data_m * norm_data_x + (1 - data_m) * Z_inf
    with torch.no_grad():
        imputed = student_gen(
            torch.tensor(X_inf,  dtype=torch.float32).to(device),
            torch.tensor(data_m, dtype=torch.float32).to(device)
        ).cpu().numpy()

    imputed = data_m * norm_data_x + (1 - data_m) * imputed
    imputed = renormalization(imputed, norm_parameters)
    imputed = rounding(imputed, data_x)
    return imputed


print("✅ Student Baseline 모델 정의 완료")

---
## 9. Light DB 메인 실험 루프 (`main_LDB.py`)

> 아래 셀에서 `DATASETS`, `MISS_RATES` 등 실험 설정을 수정하세요.


In [ ]:
import os, sys, datetime

# ── 실험 설정 ─────────────────────────────────────────────
DATASETS   = ['breast_cancer', 'spam', 'credit', 'wine', 'student']
MISS_RATES = [0.05, 0.1, 0.2, 0.3, 0.5]
SEEDS      = [42, 0, 1]       # Light DB 멀티시드 (mean+-std)
SEED       = 42               # Heavy DB 단일 시드
BATCH_SIZE = 128
HINT_RATE  = 0.9

# ── 최종 확정 하이퍼파라미터 ──────────────────────────────
GAIN_ALPHA = 100    # GAIN: 원논문(Yoon et al. 2018) 고정값
BEST_ALPHA = 50     # WGAN-FWAL / KD / Student: Tuning-A 결과
BEST_LR    = 1e-4   # Adam lr
USE_COSLR  = True   # Light DB: CosLR 수렴 가속

# ── Heavy 설정 ────────────────────────────────────────────
HEAVY_DATASETS   = ['higgs', 'criteo']
HEAVY_MISS_RATES = [0.1, 0.2, 0.3]
HEAVY_NROWS      = 200_000
HEAVY_BATCH      = 512
HEAVY_ALPHA      = 100
HEAVY_LR         = 1e-4

# ── 출력 경로 (이전 결과와 완전 분리) ────────────────────
SAVE_DIR = os.path.join('.', 'results', 'final_results_0610')
os.makedirs(SAVE_DIR, exist_ok=True)
_TS = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

# ── 로그 파일 (stdout + 파일 동시 기록) ──────────────────
LOG_PATH = os.path.join(SAVE_DIR, f'train_log_{_TS}.txt')

class _LogTee:
    """stdout 을 터미널 + 파일에 동시 기록하는 Tee wrapper."""
    def __init__(self, path, mode='w'):
        self._file   = open(path, mode, encoding='utf-8')
        self._stdout = sys.stdout
    def write(self, text):
        self._stdout.write(text)
        self._file.write(text)
        self._file.flush()
    def flush(self):
        self._stdout.flush()
        self._file.flush()
    def close(self):
        sys.stdout = self._stdout
        self._file.close()

# FID/gradient history 수집 간격
CHECKPOINT_EVERY = 100

print(f"[설정] GAIN_ALPHA={GAIN_ALPHA} | WGAN/KD/Student alpha={BEST_ALPHA} lr={BEST_LR:.0e} CosLR={USE_COSLR}")
print(f"[설정] SEEDS={SEEDS}  ->  SAVE_DIR={SAVE_DIR}")
print(f"[로그] {LOG_PATH}")


In [ ]:
# ════════════════════════════════════════════════════════
# 9. Light DB 메인 실험 루프 — 멀티시드 [42, 0, 1]
# ════════════════════════════════════════════════════════
import pandas as pd

results       = []
trained_G_all = {'light': {ds: {} for ds in DATASETS}, 'heavy': {}}
fid_hist_all  = {'light': {ds: {} for ds in DATASETS}, 'heavy': {}}
grad_hist_all = {'light': {ds: {} for ds in DATASETS}, 'heavy': {}}


def train_gain_with_history(data_x, params, grad_hist_list, fid_hist_list):
    """GAIN 학습 + (G, imputed) 반환. CHECKPOINT_EVERY마다 FID/grad 기록."""
    device     = torch.device(params.get('device', 'cpu'))
    data_m     = 1 - np.isnan(data_x)
    batch_size = params['batch_size']
    hint_rate  = params['hint_rate']
    alpha      = params['alpha']
    iterations = params['iterations']
    no, dim    = data_x.shape

    norm_data, norm_params = normalization(data_x)
    norm_data_x = np.nan_to_num(norm_data, nan=0.0)

    use_amp = params.get('use_amp', USE_AMP)
    scaler  = GradScaler(enabled=use_amp)
    G = GAINGenerator(dim, dim).to(device)
    D = GAINDiscriminator(dim, dim).to(device)
    D_opt = optim.Adam(D.parameters())
    G_opt = optim.Adam(G.parameters())

    for it in tqdm(range(iterations), desc='[GAIN] train+hist', leave=False):
        idx  = sample_batch_index(no, batch_size)
        X_mb = norm_data_x[idx]; M_mb = data_m[idx]
        Z_mb = uniform_sampler(0, 0.01, batch_size, dim)
        H_mb = M_mb * binary_sampler(hint_rate, batch_size, dim)
        X_mb = M_mb * X_mb + (1 - M_mb) * Z_mb
        Xt = torch.tensor(X_mb, dtype=torch.float32).to(device)
        Mt = torch.tensor(M_mb, dtype=torch.float32).to(device)
        Ht = torch.tensor(H_mb, dtype=torch.float32).to(device)

        D.train(); G.eval()
        with torch.no_grad():
            with autocast(enabled=use_amp): Gs = G(Xt, Mt)
        HX = Xt * Mt + Gs * (1 - Mt)
        with autocast(enabled=use_amp): Dp = D(HX.detach(), Ht)
        Dp_f32 = Dp.float()
        Dl = -torch.mean(Mt * torch.log(Dp_f32 + 1e-8) + (1 - Mt) * torch.log(1 - Dp_f32 + 1e-8))
        D_opt.zero_grad()
        scaler.scale(Dl).backward()
        scaler.step(D_opt)
        scaler.update()

        G.train(); D.eval()
        with autocast(enabled=use_amp):
            Gs = G(Xt, Mt); HX = Xt * Mt + Gs * (1 - Mt)
            Dp = D(HX, Ht)
        Dp_f32 = Dp.float(); Gs_f32 = Gs.float()
        Gl = (-torch.mean((1 - Mt) * torch.log(Dp_f32 + 1e-8))
              + alpha * torch.mean((Mt * Xt - Mt * Gs_f32) ** 2) / (torch.mean(Mt) + 1e-8))
        G_opt.zero_grad()
        scaler.scale(Gl).backward()
        scaler.step(G_opt)
        scaler.update()

        if (it + 1) % CHECKPOINT_EVERY == 0:
            grad_hist_list.append(record_gradient_magnitudes(G))
            G.eval()
            with torch.no_grad():
                Z_inf = uniform_sampler(0, 0.01, no, dim)
                Xi = torch.tensor(data_m * norm_data_x + (1 - data_m) * Z_inf,
                                  dtype=torch.float32).to(device)
                Mi = torch.tensor(data_m, dtype=torch.float32).to(device)
                imp_t = G(Xi, Mi).cpu().numpy()
            imp_t = data_m * norm_data_x + (1 - data_m) * imp_t
            fid_hist_list.append(compute_fid(norm_data_x[data_m == 0].reshape(-1, 1),
                                             imp_t[data_m == 0].reshape(-1, 1)))
            G.train()

    G.eval()
    with torch.no_grad():
        Z_inf = uniform_sampler(0, 0.01, no, dim)
        Xi = torch.tensor(data_m * norm_data_x + (1 - data_m) * Z_inf,
                          dtype=torch.float32).to(device)
        Mi = torch.tensor(data_m, dtype=torch.float32).to(device)
        imp = G(Xi, Mi).cpu().numpy()
    imp = data_m * norm_data_x + (1 - data_m) * imp
    imp = renormalization(imp, norm_params)
    imp = rounding(imp, data_x)
    return G, imp


# ── 모델별 params ──────────────────────────────────────────
_base = dict(batch_size=BATCH_SIZE, hint_rate=HINT_RATE, iterations=5000,
             device=str(DEVICE), kd_alpha=1.0, kd_temperature=3.0,
             kd_w_feat=0.5, kd_w_kl=0.5)
params_gain = dict(_base, alpha=GAIN_ALPHA, lr=1e-3,    use_coslr=False)
params_wgan = dict(_base, alpha=BEST_ALPHA, lr=BEST_LR, use_coslr=USE_COSLR)

# ── stdout -> 로그파일 동시 기록 ──────────────────────────
_log_tee = _LogTee(LOG_PATH)
sys.stdout = _log_tee

try:
    for seed in SEEDS:
        np.random.seed(seed); torch.manual_seed(seed)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
        is_first = (seed == SEEDS[0])

        print(f"\n{'='*60}")
        print(f"  SEED = {seed}  ({SEEDS.index(seed)+1}/{len(SEEDS)})")
        print(f"{'='*60}")

        for ds in DATASETS:
            for miss_rate in MISS_RATES:
                print(f"\n{'-'*60}")
                print(f"  Dataset: {ds} | Miss Rate: {miss_rate} | Seed: {seed}")
                print(f"{'-'*60}")
                ori_data, miss_data, data_m = get_gain_data(ds, miss_rate, seed)
                g_hist_gain = []; f_hist_gain = []

                # GAIN (alpha=100, 원논문 고정)
                G_gain, imp_gain = train_gain_with_history(
                    miss_data, params_gain, g_hist_gain, f_hist_gain)
                m = evaluate_all(ori_data, imp_gain, data_m)
                m.update({'Model':'GAIN','Dataset':ds,'Miss_Rate':miss_rate,
                          'Seed':seed,'DB_Type':'light'})
                results.append(m)
                print(f"  [{'GAIN':15s}] RMSE={m['rmse']:.4f}  MAE={m['mae']:.4f}  "
                      f"AUROC={m['auroc']:.4f}  PredAUC={m['pred_auc']:.4f}")

                # WGAN_FWAL (alpha=50, CosLR, 실제 G 반환)
                imp_fwal, G_fwal = wganwithFWAL(miss_data, params_wgan)
                m = evaluate_all(ori_data, imp_fwal, data_m)
                m.update({'Model':'WGAN_FWAL','Dataset':ds,'Miss_Rate':miss_rate,
                          'Seed':seed,'DB_Type':'light'})
                results.append(m)
                print(f"  [{'WGAN_FWAL':15s}] RMSE={m['rmse']:.4f}  MAE={m['mae']:.4f}  "
                      f"AUROC={m['auroc']:.4f}  PredAUC={m['pred_auc']:.4f}")

                # KD (real FWALGenerator teacher)
                imp_kd, G_kd = knowledge_distillation(miss_data, params_wgan)
                m = evaluate_all(ori_data, imp_kd, data_m)
                m.update({'Model':'KD','Dataset':ds,'Miss_Rate':miss_rate,
                          'Seed':seed,'DB_Type':'light'})
                results.append(m)
                print(f"  [{'KD':15s}] RMSE={m['rmse']:.4f}  MAE={m['mae']:.4f}  "
                      f"AUROC={m['auroc']:.4f}  PredAUC={m['pred_auc']:.4f}")

                # Student Baseline
                imp_sb = student_baseline(miss_data, params_wgan)
                m = evaluate_all(ori_data, imp_sb, data_m)
                m.update({'Model':'Student_Baseline','Dataset':ds,'Miss_Rate':miss_rate,
                          'Seed':seed,'DB_Type':'light'})
                results.append(m)
                print(f"  [{'Student_Baseline':15s}] RMSE={m['rmse']:.4f}  MAE={m['mae']:.4f}  "
                      f"AUROC={m['auroc']:.4f}  PredAUC={m['pred_auc']:.4f}")

                # 첫 번째 시드에서만 Generator/history 저장
                if is_first:
                    if miss_rate not in trained_G_all['light'][ds]:
                        trained_G_all['light'][ds][miss_rate] = {}
                    norm_d, norm_p = normalization(ori_data)
                    trained_G_all['light'][ds][miss_rate].update({
                        'GAIN': G_gain, 'WGAN_FWAL': G_fwal, 'KD': G_kd,
                        '_ori': ori_data, '_mask': data_m,
                        '_imp': {'GAIN': imp_gain, 'WGAN_FWAL': imp_fwal, 'KD': imp_kd},
                        '_norm_x': np.nan_to_num(norm_d, nan=0.0),
                        '_norm_p': norm_p,
                    })
                    fid_hist_all['light'][ds][miss_rate]  = {'GAIN': f_hist_gain, 'WGAN_FWAL': []}
                    grad_hist_all['light'][ds][miss_rate] = {'GAIN': g_hist_gain, 'WGAN_FWAL': getattr(G_fwal, '_grad_hist', [])}

    _csv_light = os.path.join(SAVE_DIR, f'light_final_{_TS}.csv')
    pd.DataFrame(results).to_csv(_csv_light, index=False, encoding='utf-8-sig')
    print(f"\n[저장] {_csv_light}")
    print(f"\n\u2705 Light DB 멀티시드 실험 완료! ({len(results)}개 레코드)")

finally:
    sys.stdout = _log_tee._stdout
    _log_tee._file.flush()


---
## 10. Heavy DB 메인 실험 루프

> HIGGS (`../data/HIGGS.csv`) 및 Criteo (`../criteoDB/train.txt`) 파일이 있을 때만 실행하세요.  
> `HEAVY_NROWS` 로 메모리 사용량을 조절하세요 (기본 200,000).


In [ ]:
# ── Heavy DB 실험 설정 ────────────────────────────────────
HEAVY_PARAMS_WGAN = {
    'batch_size'    : HEAVY_BATCH,
    'hint_rate'     : HINT_RATE,
    'alpha'         : HEAVY_ALPHA,   # 100
    'iterations'    : 5000,
    'lr'            : HEAVY_LR,      # 1e-4
    'use_coslr'     : False,         # Heavy 200K행: CosLR 발산 -> 비적용
    'device'        : str(DEVICE),
    'kd_alpha'      : 1.0,
    'kd_temperature': 3.0,
    'kd_w_feat'     : 0.5,
    'kd_w_kl'       : 0.5,
}
HEAVY_PARAMS_GAIN = dict(HEAVY_PARAMS_WGAN, alpha=GAIN_ALPHA, lr=1e-3)

print(f"[Heavy] WGAN alpha={HEAVY_ALPHA} lr={HEAVY_LR:.0e} CosLR=False")
print(f"[Heavy] GAIN  alpha={GAIN_ALPHA}")


In [ ]:
# ════════════════════════════════════════════════════════
# 10. Heavy DB 메인 실험 루프
# ════════════════════════════════════════════════════════
import pandas as pd

results_heavy = []

_log_tee_h = _LogTee(os.path.join(SAVE_DIR, f'train_log_heavy_{_TS}.txt'))
sys.stdout  = _log_tee_h

try:
    for ds_name in HEAVY_DATASETS:
        chk_path = HIGGS_PATH if ds_name == 'higgs' else CRITEO_TRAIN
        if not os.path.exists(chk_path):
            print(f'  [{ds_name.upper()}] 파일 없음 -> 건너뜀')
            continue

        trained_G_all['heavy'][ds_name] = {}
        fid_hist_all ['heavy'][ds_name] = {}
        grad_hist_all['heavy'][ds_name] = {}

        for miss_rate in HEAVY_MISS_RATES:
            print(f"\n{'='*60}")
            print(f"  [HEAVY] {ds_name.upper()} | Miss Rate: {miss_rate:.2f}")
            print(f"{'='*60}")

            loader = get_heavy_dataloader(
                dataset_name=ds_name, missing_rate=miss_rate,
                seed=SEED, batch_size=HEAVY_NROWS, nrows=HEAVY_NROWS)
            X_orig_t, _, mask_t, _ = next(iter(loader))
            ori_h  = X_orig_t.numpy().astype(np.float64)
            mask_h = mask_t.numpy().astype(np.float64)
            miss_h = ori_h.copy(); miss_h[mask_h == 0] = np.nan
            dim_h  = ori_h.shape[1]
            g_hist_gain = []; f_hist_gain = []

            def _eval_log(model_name, imp):
                try:
                    m = evaluate_all(ori_h, imp, mask_h)
                except Exception as e:
                    print(f'  [{model_name}] ERROR: {e}')
                    m = {'rmse': float('nan'), 'mae': float('nan'),
                         'auroc': float('nan'), 'pred_auc': float('nan')}
                m.update({'Model': model_name, 'Dataset': ds_name.upper(),
                          'Miss_Rate': miss_rate, 'DB_Type': 'heavy'})
                results_heavy.append(m)
                print(f"  [{model_name:15s}] RMSE={m['rmse']:.4f}  MAE={m['mae']:.4f}  "
                      f"AUROC={m['auroc']:.4f}  PredAUC={m['pred_auc']:.4f}")
                return m

            # GAIN
            G_gain_h, imp_gain_h = train_gain_with_history(
                miss_h, HEAVY_PARAMS_GAIN, g_hist_gain, f_hist_gain)
            _eval_log('GAIN', imp_gain_h)

            # WGAN_FWAL (실제 G 반환)
            imp_fwal_h, G_fwal_h = wganwithFWAL(miss_h, HEAVY_PARAMS_WGAN)
            _eval_log('WGAN_FWAL', imp_fwal_h)

            # KD (real teacher)
            imp_kd_h, G_kd_h = knowledge_distillation(miss_h, HEAVY_PARAMS_WGAN)
            _eval_log('KD', imp_kd_h)

            # Student Baseline
            imp_sb_h = student_baseline(miss_h, HEAVY_PARAMS_WGAN)
            _eval_log('Student_Baseline', imp_sb_h)

            _norm_h_data, _norm_h_params = normalization(ori_h)
            trained_G_all['heavy'][ds_name][miss_rate] = {
                'GAIN': G_gain_h, 'WGAN_FWAL': G_fwal_h, 'KD': G_kd_h,
                '_ori': ori_h, '_mask': mask_h,
                '_imp': {'GAIN': imp_gain_h, 'WGAN_FWAL': imp_fwal_h, 'KD': imp_kd_h},
                '_norm_x': np.nan_to_num(_norm_h_data, nan=0.0),
                '_norm_p': _norm_h_params,
            }
            fid_hist_all ['heavy'][ds_name][miss_rate] = {'GAIN': f_hist_gain, 'WGAN_FWAL': []}
            grad_hist_all['heavy'][ds_name][miss_rate] = {'GAIN': g_hist_gain, 'WGAN_FWAL': []}

    if results_heavy:
        _csv_heavy = os.path.join(SAVE_DIR, f'heavy_final_{_TS}.csv')
        pd.DataFrame(results_heavy).to_csv(_csv_heavy, index=False, encoding='utf-8-sig')
        print(f"\n[저장] {_csv_heavy}")

    print("\n\u2705 Heavy DB 실험 완료!")

finally:
    sys.stdout = _log_tee_h._stdout
    _log_tee_h._file.flush()


---
## 11. 결과 요약 및 시각화


In [ ]:
# ════════════════════════════════════════════════════════
# 결과 요약 — mean±std + 이미지 + CSV 저장 (final_results_0610/)
# ════════════════════════════════════════════════════════
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

summary_df  = pd.DataFrame(results)
MODEL_ORDER = ['GAIN', 'WGAN_FWAL', 'KD', 'Student_Baseline']
METRICS_4   = ['rmse', 'mae', 'auroc', 'pred_auc']
seeds_str   = ', '.join(str(s) for s in sorted(summary_df['Seed'].unique())) \
              if 'Seed' in summary_df.columns else str(SEEDS)

# ── mean ± std 텍스트 출력 ─────────────────────────────────
agg = {c: ['mean', 'std'] for c in METRICS_4 if c in summary_df.columns}
sdf = summary_df.groupby(['Model','Dataset','Miss_Rate']).agg(agg).round(4)
sdf.columns = ['_'.join(c) for c in sdf.columns]
sdf = sdf.reset_index()
for c in METRICS_4:
    if f'{c}_mean' in sdf.columns:
        sdf[f'{c}_fmt'] = (sdf[f'{c}_mean'].map('{:.4f}'.format)
                            + ' +- ' + sdf[f'{c}_std'].map('{:.4f}'.format))

print("\n[Light DB] mean +- std  (3 seeds: 42, 0, 1)")
print(sdf[['Model','Dataset','Miss_Rate']
          + [f'{c}_fmt' for c in METRICS_4 if f'{c}_fmt' in sdf.columns]]
      .to_string(index=False))

# ── 통합 CSV 저장 (Light + Heavy) ──────────────────────────
all_rows = list(results) + (results_heavy if results_heavy else [])
all_df   = pd.DataFrame(all_rows)
_csv_all = os.path.join(SAVE_DIR, f'all_results_{_TS}.csv')
all_df.to_csv(_csv_all, index=False, encoding='utf-8-sig')
print(f"\n[저장] 통합 CSV -> {_csv_all}")

# mean/std per seed도 별도 CSV 저장
_csv_std = os.path.join(SAVE_DIR, f'light_std_{_TS}.csv')
sdf.to_csv(_csv_std, index=False, encoding='utf-8-sig')
print(f"[저장] mean+-std CSV -> {_csv_std}")

DS_ORDER = ['breast_cancer','spam','credit','wine','student']
DS_LABEL = {'breast_cancer':'Breast','spam':'Spam','credit':'Credit',
             'wine':'Wine','student':'Student'}

# ── 이미지 1: miss_rate=0.2  셀마다 mean\n+-std ─────────────
for metric in ['rmse', 'mae']:
    sub02 = summary_df[summary_df['Miss_Rate'] == 0.2]
    grp02 = (sub02.groupby(['Model','Dataset'])[metric]
             .agg(['mean','std']).reset_index())
    grp02['val'] = (grp02['mean'].map('{:.4f}'.format)
                    + '\n+-' + grp02['std'].map('{:.4f}'.format))
    pivot_mean = grp02.pivot(index='Model', columns='Dataset', values='mean')
    pivot_val  = grp02.pivot(index='Model', columns='Dataset', values='val')
    pivot_mean = pivot_mean.reindex(MODEL_ORDER)[DS_ORDER]
    pivot_val  = pivot_val.reindex(MODEL_ORDER)[DS_ORDER]

    cell_text   = []
    cell_colors = [['white'] * len(DS_ORDER) for _ in MODEL_ORDER]
    for m_name in MODEL_ORDER:
        row = []
        for d in DS_ORDER:
            v = pivot_val.loc[m_name, d] if (m_name in pivot_val.index and d in pivot_val.columns) else 'N/A'
            row.append(str(v) if not (isinstance(v, float) and np.isnan(v)) else 'N/A')
        cell_text.append(row)
    for j in range(len(DS_ORDER)):
        vals = []
        for m in MODEL_ORDER:
            if m in pivot_mean.index and DS_ORDER[j] in pivot_mean.columns:
                v = pivot_mean.loc[m, DS_ORDER[j]]
                vals.append(float('nan') if isinstance(v, float) and np.isnan(v) else v)
            else:
                vals.append(float('nan'))
        valid = [(i, v) for i, v in enumerate(vals) if not np.isnan(v)]
        if valid:
            cell_colors[min(valid, key=lambda x: x[1])[0]][j] = '#c8f7c5'

    fig, ax = plt.subplots(figsize=(14, 4.8)); ax.axis('off')
    tbl = ax.table(cellText=cell_text, rowLabels=MODEL_ORDER,
                   colLabels=[DS_LABEL[d] for d in DS_ORDER],
                   cellColours=cell_colors, cellLoc='center', loc='center')
    tbl.auto_set_font_size(False); tbl.set_fontsize(8.5); tbl.scale(1.0, 3.2)
    for (r, c), cell in tbl.get_celld().items():
        cell.set_edgecolor('#aaaaaa')
        if r == 0: cell.set_facecolor('#2c3e50'); cell.set_text_props(color='white', fontweight='bold')
        if c == -1: cell.set_facecolor('#ecf0f1'); cell.set_text_props(fontweight='bold', fontsize=8)
    ax.set_title(
        f'Light DB  {metric.upper()}  mean +- std  |  miss_rate = 0.2  (seeds: {seeds_str})\n'
        f'GAIN alpha={GAIN_ALPHA}  /  WGAN·KD·Student alpha={BEST_ALPHA}  lr={BEST_LR:.0e}  CosLR={USE_COSLR}',
        fontsize=10, fontweight='bold', pad=10)
    plt.tight_layout()
    _img = os.path.join(SAVE_DIR, f'table_{metric}_miss02_std_{_TS}.png')
    plt.savefig(_img, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()
    print(f"[저장] {_img}")

# ── 이미지 2: 전체 결측률 × 데이터셋  mean+-std 표 ─────────
for metric in ['rmse', 'mae']:
    _grp = (summary_df.groupby(['Model','Dataset','Miss_Rate'])[metric]
            .agg(['mean','std']).reset_index())
    _grp['val'] = _grp['mean'].map('{:.3f}'.format) + '\n+-' + _grp['std'].map('{:.3f}'.format)
    MR_ORDER    = sorted(_grp['Miss_Rate'].unique())
    col_labels2 = [f'{d[:4]}\n{int(mr*100)}%' for d in DS_ORDER for mr in MR_ORDER]
    cell_text2  = []
    for m_name in MODEL_ORDER:
        row = []
        for d in DS_ORDER:
            for mr in MR_ORDER:
                sub = _grp[(_grp['Model']==m_name)&(_grp['Dataset']==d)&(_grp['Miss_Rate']==mr)]
                row.append(sub['val'].values[0] if len(sub) else 'N/A')
        cell_text2.append(row)

    n_cols = len(col_labels2)
    fig2, ax2 = plt.subplots(figsize=(max(22, n_cols * 1.3), len(MODEL_ORDER) * 1.5 + 2.0))
    ax2.axis('off')
    tbl2 = ax2.table(cellText=cell_text2, rowLabels=MODEL_ORDER, colLabels=col_labels2,
                     cellLoc='center', rowLoc='center', loc='center')
    tbl2.auto_set_font_size(False); tbl2.set_fontsize(7.5); tbl2.scale(1, 3.0)
    kd_i = MODEL_ORDER.index('KD')
    for j in range(n_cols): tbl2[(kd_i+1, j)].set_facecolor('#FFF9C4')
    for j in range(n_cols): tbl2[(0, j)].set_text_props(fontweight='bold')
    ax2.set_title(
        f'Light DB  {metric.upper()}  mean +- std  (seeds: {seeds_str}, all miss_rates)\n'
        f'GAIN alpha={GAIN_ALPHA}  /  WGAN·KD·Student alpha={BEST_ALPHA}  lr={BEST_LR:.0e}  CosLR={USE_COSLR}',
        fontsize=11, fontweight='bold', pad=12)
    plt.tight_layout()
    _img2 = os.path.join(SAVE_DIR, f'table_{metric}_all_std_{_TS}.png')
    plt.savefig(_img2, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()
    print(f"[저장] {_img2}")

# ── Heavy 결과 출력 + 이미지 ─────────────────────────────────
if results_heavy:
    heavy_df = pd.DataFrame(results_heavy)
    print(f"\n[Heavy DB Results]  (seed={SEED}, 단일 실행)")
    print(heavy_df[['Model','Dataset','Miss_Rate','rmse','mae','auroc']].to_string(index=False))
    for metric in ['rmse', 'mae']:
        h_ds   = sorted(heavy_df['Dataset'].unique())
        h_mrs  = sorted(heavy_df['Miss_Rate'].unique())
        h_cols = [f'{d}\n{int(mr*100)}%' for d in h_ds for mr in h_mrs]
        h_text = []
        for m_name in MODEL_ORDER:
            row = []
            for d in h_ds:
                for mr in h_mrs:
                    sub = heavy_df[(heavy_df['Model']==m_name)&(heavy_df['Dataset']==d)&(heavy_df['Miss_Rate']==mr)]
                    row.append(f'{sub[metric].values[0]:.4f}' if len(sub) else 'N/A')
            h_text.append(row)
        fig_h, ax_h = plt.subplots(figsize=(max(12, len(h_cols)*2), 4))
        ax_h.axis('off')
        tbl_h = ax_h.table(cellText=h_text, rowLabels=MODEL_ORDER, colLabels=h_cols,
                           cellLoc='center', loc='center')
        tbl_h.auto_set_font_size(False); tbl_h.set_fontsize(8.5); tbl_h.scale(1.0, 2.4)
        for (r, c), cell in tbl_h.get_celld().items():
            cell.set_edgecolor('#aaaaaa')
            if r == 0: cell.set_facecolor('#34495e'); cell.set_text_props(color='white', fontweight='bold')
            if c == -1: cell.set_facecolor('#ecf0f1'); cell.set_text_props(fontweight='bold')
        ax_h.set_title(f'Heavy DB  {metric.upper()}  (seed={SEED})',
                       fontsize=11, fontweight='bold', pad=10)
        plt.tight_layout()
        _img_h = os.path.join(SAVE_DIR, f'table_heavy_{metric}_{_TS}.png')
        plt.savefig(_img_h, dpi=150, bbox_inches='tight')
        plt.show(); plt.close()
        print(f"[저장] {_img_h}")

print(f"\n총 실험 레코드: {len(all_df)}  "
      f"(Light={len(summary_df)}, Heavy={len(results_heavy) if results_heavy else 0})")
print(f"저장 폴더: {os.path.abspath(SAVE_DIR)}")


In [ ]:
# ════════════════════════════════════════════════════════
# ② Missing rate별 RMSE/MAE Figure — Light DB
#    모델별 mean±std errorbar, SAVE_DIR 저장
# ════════════════════════════════════════════════════════
METRICS_TO_PLOT = ['rmse', 'mae']
METRIC_LABELS   = {'rmse': 'RMSE ↓', 'mae': 'MAE ↓'}
COLORS = {'GAIN': 'steelblue', 'WGAN_FWAL': 'seagreen',
          'KD': 'darkorange', 'Student_Baseline': 'crimson'}

seeds_str2 = (', '.join(str(s) for s in sorted(summary_df['Seed'].unique()))
              if 'Seed' in summary_df.columns else '42, 0, 1')

for metric in METRICS_TO_PLOT:
    fig, axes = plt.subplots(1, len(DATASETS), figsize=(5 * len(DATASETS), 4), squeeze=False)
    for col_j, ds in enumerate(DATASETS):
        ax = axes[0][col_j]
        for mname in MODEL_ORDER:
            sub = summary_df[(summary_df['Dataset'] == ds) & (summary_df['Model'] == mname)]
            if sub.empty:
                continue
            grp = sub.groupby('Miss_Rate')[metric].agg(['mean', 'std']).reset_index()
            ax.errorbar(grp['Miss_Rate'], grp['mean'], yerr=grp['std'],
                        marker='o', label=mname, color=COLORS.get(mname),
                        linewidth=2, capsize=3, elinewidth=1.2)
        ax.set_title(ds, fontsize=10, fontweight='bold')
        ax.set_xlabel('Missing Rate')
        ax.set_ylabel(METRIC_LABELS[metric] if col_j == 0 else '')
        if col_j == 0:
            ax.legend(fontsize=7)
        ax.grid(alpha=0.3)
    plt.suptitle(
        f'Light DB — {metric.upper()} by Missing Rate  (mean ± std, seeds: {seeds_str2})',
        fontsize=12, fontweight='bold', y=1.02)
    plt.tight_layout()
    _save30 = os.path.join(SAVE_DIR, f'fig_missrate_{metric}_{_TS}.png')
    plt.savefig(_save30, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()
    print(f'[저장] {_save30}')


In [ ]:
# ════════════════════════════════════════════════════════
# ② Missing rate 히트맵 + Heavy DB 시각화 (SAVE_DIR 저장)
# ════════════════════════════════════════════════════════
try:
    import seaborn as sns
except ImportError:
    sns = None

METRICS_TO_PLOT = ['rmse', 'mae']
METRIC_LABELS   = {'rmse': 'RMSE ↓', 'mae': 'MAE ↓'}

# ── Light DB 히트맵: 모델 × 결측률 (평균) ──────────────
if sns is not None:
    fig, axes = plt.subplots(1, len(METRICS_TO_PLOT),
                             figsize=(7 * len(METRICS_TO_PLOT), 4))
    if len(METRICS_TO_PLOT) == 1:
        axes = [axes]
    for ax, metric in zip(axes, METRICS_TO_PLOT):
        pivot = summary_df.groupby(['Model', 'Miss_Rate'])[metric].mean().unstack()
        pivot = pivot.reindex(MODEL_ORDER)
        sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlOrRd',
                    linewidths=0.5, ax=ax, cbar_kws={'label': metric})
        ax.set_title(f'Avg {METRIC_LABELS[metric]}')
        ax.set_xlabel('Missing Rate'); ax.set_ylabel('Model')
    plt.suptitle('Light DB — Average Metrics by Model & Missing Rate',
                 fontsize=12, fontweight='bold', y=1.03)
    plt.tight_layout()
    _save_hm = os.path.join(SAVE_DIR, f'fig_heatmap_light_{_TS}.png')
    plt.savefig(_save_hm, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()
    print(f'[저장] {_save_hm}')
else:
    print('[SKIP] seaborn 없음 — 히트맵 건너뜀')

# ── Heavy DB 결과 테이블 출력 ──────────────────────────
if results_heavy:
    heavy_df = pd.DataFrame(results_heavy)
    for metric in METRICS_TO_PLOT:
        print(f'\n── Heavy DB {METRIC_LABELS[metric]} ──')
        pivot_h = heavy_df.pivot_table(
            index=['Dataset', 'Miss_Rate'], columns='Model', values=metric
        ).round(4)
        print(pivot_h)

    # Heavy DB 라인 차트
    heavy_ds_list = sorted(heavy_df['Dataset'].unique().tolist())
    for metric in METRICS_TO_PLOT:
        fig, axes = plt.subplots(1, len(heavy_ds_list),
                                 figsize=(6 * len(heavy_ds_list), 4), squeeze=False)
        for col_j, ds in enumerate(heavy_ds_list):
            ax = axes[0][col_j]
            sub = heavy_df[heavy_df['Dataset'] == ds]
            for mname, grp in sub.groupby('Model'):
                ax.plot(grp['Miss_Rate'], grp[metric], marker='o',
                        label=mname, color=COLORS.get(mname), linewidth=2)
            ax.set_title(ds, fontsize=10)
            ax.set_xlabel('Missing Rate')
            ax.set_ylabel(METRIC_LABELS[metric] if col_j == 0 else '')
            if col_j == 0:
                ax.legend(fontsize=7)
            ax.grid(alpha=0.3)
        plt.suptitle(f'Heavy DB — {metric.upper()} by Missing Rate',
                     fontsize=12, fontweight='bold', y=1.02)
        plt.tight_layout()
        _save_hv = os.path.join(SAVE_DIR, f'fig_heavy_{metric}_{_TS}.png')
        plt.savefig(_save_hv, dpi=150, bbox_inches='tight')
        plt.show(); plt.close()
        print(f'[저장] {_save_hv}')
else:
    print('Heavy DB 결과 없음')


---
## 12. 고급 평가 — FLOPs / Latency / FID / Diversity

In [ ]:
# ════════════════════════════════════════════════════════
# 12. 고급 평가 — Light & Heavy 전체 조합 일괄 수행
# ════════════════════════════════════════════════════════

ADV_SAVE_DIR = SAVE_DIR  # all outputs to final_results_0610/
device_adv   = torch.device(str(DEVICE))

all_adv_records = []   # 전체 결과를 DataFrame으로 저장용

def run_one_adv_eval(db_type, ds_name, miss_rate, slot):
    """
    slot = trained_G_all[db_type][ds_name][miss_rate]
    고급 지표를 계산하고 딕셔너리로 반환.
    """
    ori      = slot['_ori']
    mask     = slot['_mask']
    norm_x   = slot['_norm_x']
    norm_p   = slot['_norm_p']
    imp_dict = slot['_imp']
    fid_h    = fid_hist_all [db_type][ds_name][miss_rate]
    grad_h   = grad_hist_all[db_type][ds_name][miss_rate]

    label  = f'[{db_type.upper()}] {ds_name} | miss={miss_rate}'
    print(f'\n{"─"*60}')
    print(f'  {label}')
    print(f'{"─"*60}')

    summary = {}
    for mname, G in slot.items():
        if mname.startswith('_'):
            continue
        imp = imp_dict.get(mname)
        if imp is None:
            continue

        norm_imp, _ = normalization(imp, norm_p)
        lat_mean, lat_std = measure_latency(G, norm_x, mask, device_adv, runs=20)
        div_score         = diversity_score(ori, imp, mask)

        summary[mname] = {
            'db_type'  : db_type,
            'dataset'  : ds_name,
            'miss_rate': miss_rate,
            'model'    : mname,
            'params'   : count_parameters(G),
            'flops'    : count_flops(G),
            'latency'  : lat_mean,
            'lat_std'  : lat_std,
            'diversity': div_score,
            'fid_final': fid_h[mname][-1] if fid_h.get(mname) else None,
        }

    # 텍스트 테이블
    model_names = list(summary.keys())
    header = f"  {'Metric':28s}" + ''.join(f'{m:>18s}' for m in model_names)
    sep    = '─' * len(header)
    print(sep); print(header); print(sep)
    for key, label_str in [
        ('params',    'Parameters'),
        ('flops',     'FLOPs (approx)'),
        ('latency',   'Latency mean (ms)'),
        ('lat_std',   'Latency std (ms)'),
        ('diversity', 'Diversity (bins ↓)'),
        ('fid_final', 'Final FID ↓'),
    ]:
        row = f'  {label_str:28s}'
        for m in model_names:
            val = summary[m][key]
            row += f'{val:>18.3f}' if isinstance(val, float) else \
                   f'{val:>18,}'   if isinstance(val, int)   else f"{'N/A':>18s}"
        print(row)
    print(sep)

    # 시각화 저장
    save_dir = os.path.join(ADV_SAVE_DIR, db_type, ds_name, str(miss_rate))
    os.makedirs(save_dir, exist_ok=True)

    plot_efficiency_bars(
        {m: {k: summary[m][k] for k in ('params','flops','latency','diversity')}
         for m in model_names},
        title=f'Efficiency — {ds_name} (miss={miss_rate})',
        save_path=os.path.join(save_dir, 'efficiency_bars.png')
    )

    if fid_h.get('GAIN'):
        plot_fid_convergence_single(
            fid_h['GAIN'],
            title=f'GAIN FID Convergence — {ds_name} (miss={miss_rate})',
            save_path=os.path.join(save_dir, 'fid_convergence.png')
        )

    return list(summary.values())


# ── Light DB 전체 순회 ──────────────────────────────────
print('\n' + '='*60)
print('  [LIGHT DB] 고급 평가 시작')
print('='*60)
for ds in DATASETS:
    for mr in MISS_RATES:
        if ds in trained_G_all['light'] and mr in trained_G_all['light'][ds]:
            records = run_one_adv_eval('light', ds, mr,
                                       trained_G_all['light'][ds][mr])
            all_adv_records.extend(records)

# ── Heavy DB 전체 순회 ─────────────────────────────────
print('\n' + '='*60)
print('  [HEAVY DB] 고급 평가 시작')
print('='*60)
for ds_name in trained_G_all['heavy']:
    for mr in HEAVY_MISS_RATES:
        if mr in trained_G_all['heavy'][ds_name]:
            records = run_one_adv_eval('heavy', ds_name, mr,
                                       trained_G_all['heavy'][ds_name][mr])
            all_adv_records.extend(records)

# ── 통합 요약 DataFrame ────────────────────────────────
adv_df = pd.DataFrame(all_adv_records)
print('\n\n' + '='*60)
print('  [고급 평가 전체 요약]')
print('='*60)
print(adv_df[['db_type','dataset','miss_rate','model',
              'params','flops','latency','diversity','fid_final']]
      .to_string(index=False))

# ③ Efficiency 표 CSV 저장
_csv_eff = os.path.join(SAVE_DIR, f'efficiency_{_TS}.csv')
adv_df.to_csv(_csv_eff, index=False, encoding='utf-8-sig')
print(f'\n[저장] Efficiency CSV -> {_csv_eff}')

# ── FID 통합 시각화 ────────────────────────────────────

# Light DB: 데이터셋별 결측률 FID 수렴 곡선
print('\n[FID] Light DB — 데이터셋별 결측률 FID 수렴 곡선')
for ds in DATASETS:
    if ds not in fid_hist_all['light']:
        continue
    fid_by_mr = {
        mr: fid_hist_all['light'][ds][mr].get('GAIN', [])
        for mr in MISS_RATES
        if mr in fid_hist_all['light'][ds]
    }
    plot_fid_by_miss_rate(
        fid_by_mr,
        miss_rates=MISS_RATES,
        title=f'GAIN FID Convergence by Miss Rate — {ds}',
        save_path=os.path.join(ADV_SAVE_DIR, 'light', ds, 'fid_by_miss_rate.png')
    )

# Light DB: 최종 FID 히트맵
print('\n[FID] Light DB — 최종 FID 히트맵 (데이터셋 × 결측률)')
fid_final_light = {}
for ds in DATASETS:
    fid_final_light[ds] = {}
    for mr in MISS_RATES:
        hist = fid_hist_all.get('light', {}).get(ds, {}).get(mr, {}).get('GAIN', [])
        fid_final_light[ds][mr] = hist[-1] if hist else None

plot_fid_final_heatmap(
    fid_final_light,
    title='GAIN Final FID — Light DB (Dataset × Miss Rate)',
    save_path=os.path.join(ADV_SAVE_DIR, 'light', 'fid_final_heatmap.png')
)

# Heavy DB: 최종 FID 히트맵
if trained_G_all['heavy']:
    print('\n[FID] Heavy DB — 최종 FID 히트맵')
    fid_final_heavy = {}
    for ds_name in trained_G_all['heavy']:
        fid_final_heavy[ds_name] = {}
        for mr in HEAVY_MISS_RATES:
            hist = fid_hist_all.get('heavy', {}).get(ds_name, {}).get(mr, {}).get('GAIN', [])
            fid_final_heavy[ds_name][mr] = hist[-1] if hist else None

    plot_fid_final_heatmap(
        fid_final_heavy,
        title='GAIN Final FID — Heavy DB (Dataset × Miss Rate)',
        save_path=os.path.join(ADV_SAVE_DIR, 'heavy', 'fid_final_heatmap.png')
    )



# ════════════════════════════════════════════════════════
# ⑤ Gradient Magnitude Figure — GAIN vs WGAN-FWAL
# ════════════════════════════════════════════════════════
print('\n[Grad Magnitude] GAIN vs WGAN-FWAL 비교')

_mr_target = 0.2  # 대표 결측률
all_gain_grads = []; all_wgan_grads = []

for ds in DATASETS:
    gh = grad_hist_all.get('light', {}).get(ds, {}).get(_mr_target, {})
    g = [float(np.mean(list(d.values()))) for d in gh.get('GAIN', []) if d]
    w = [float(np.mean(list(d.values()))) for d in gh.get('WGAN_FWAL', []) if d]
    all_gain_grads.append(g)
    all_wgan_grads.append(w)
    if g or w:
        # Per-dataset figure
        steps_g = [(i+1)*CHECKPOINT_EVERY for i in range(len(g))]
        steps_w = [(i+1)*CHECKPOINT_EVERY for i in range(len(w))]
        fig_gd, ax_gd = plt.subplots(figsize=(7, 4))
        if g:
            ax_gd.plot(steps_g, g, marker='o', label='GAIN', color='steelblue', linewidth=2)
        if w:
            ax_gd.plot(steps_w, w, marker='s', label='WGAN-FWAL', color='seagreen', linewidth=2)
        ax_gd.set_title(f'Gradient Magnitude — {ds} (miss={_mr_target})',
                        fontsize=11, fontweight='bold')
        ax_gd.set_xlabel('Training Iteration'); ax_gd.set_ylabel('Avg Gradient Norm')
        ax_gd.legend(); ax_gd.grid(alpha=0.3)
        plt.tight_layout()
        _gp = os.path.join(SAVE_DIR, 'light', ds, f'grad_mag_miss{int(_mr_target*100)}.png')
        os.makedirs(os.path.dirname(_gp), exist_ok=True)
        plt.savefig(_gp, dpi=150, bbox_inches='tight')
        plt.show(); plt.close()
        print(f'  [저장] {_gp}')

# Summary: averaged across datasets
n_g = max((len(g) for g in all_gain_grads if g), default=0)
n_w = max((len(w) for w in all_wgan_grads if w), default=0)
mean_gain_g = [float(np.mean([g[i] for g in all_gain_grads if i < len(g)])) for i in range(n_g)]
mean_wgan_g = [float(np.mean([w[i] for w in all_wgan_grads if i < len(w)])) for i in range(n_w)]

if mean_gain_g or mean_wgan_g:
    fig_gm, ax_gm = plt.subplots(figsize=(8, 5))
    if mean_gain_g:
        ax_gm.plot([(i+1)*CHECKPOINT_EVERY for i in range(n_g)], mean_gain_g,
                   marker='o', label='GAIN', color='steelblue', linewidth=2)
    if mean_wgan_g:
        ax_gm.plot([(i+1)*CHECKPOINT_EVERY for i in range(n_w)], mean_wgan_g,
                   marker='s', label='WGAN-FWAL', color='seagreen', linewidth=2)
    ax_gm.set_title(
        f'Gradient Magnitude: GAIN vs WGAN-FWAL\n'
        f'(miss_rate={_mr_target}, averaged over {len(DATASETS)} datasets)',
        fontsize=12, fontweight='bold')
    ax_gm.set_xlabel('Training Iteration'); ax_gm.set_ylabel('Avg Gradient Norm')
    ax_gm.legend(fontsize=10); ax_gm.grid(alpha=0.3)
    plt.tight_layout()
    _gm_path = os.path.join(SAVE_DIR, f'gradient_magnitude_summary_{_TS}.png')
    plt.savefig(_gm_path, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()
    print(f'[저장] Gradient Magnitude Summary -> {_gm_path}')
else:
    print('  [INFO] grad_hist_all 데이터 없음 (학습 후 실행 필요)')

print(f'\n결과 이미지 저장 경로: {os.path.abspath(SAVE_DIR)}')


In [ ]:
# ── 최종 FID 3모델 비교 바 차트 ───────────────────────────

def compute_final_fid_all(trained_G_all, fid_hist_all, db_type, datasets, miss_rates):
    """
    GAIN: 이미 수집된 fid_hist의 마지막 값 사용
    WGAN_FWAL / KD: imputed 결과로 최종 FID 직접 계산
    """
    records = []
    for ds in datasets:
        if ds not in trained_G_all[db_type]:
            continue
        for mr in miss_rates:
            if mr not in trained_G_all[db_type][ds]:
                continue
            slot   = trained_G_all[db_type][ds][mr]
            norm_p = slot['_norm_p']
            norm_x = slot['_norm_x']
            mask   = slot['_mask']

            for mname in ('GAIN', 'WGAN_FWAL', 'KD'):
                # GAIN은 fid_hist 마지막 값, 나머지는 직접 계산
                if mname == 'GAIN':
                    hist = fid_hist_all[db_type][ds][mr].get('GAIN', [])
                    fid_val = hist[-1] if hist else None
                else:
                    imp = slot['_imp'].get(mname)
                    if imp is None:
                        fid_val = None
                    else:
                        imp_norm, _ = normalization(imp, norm_p)
                        real_vals = norm_x[mask == 0].reshape(-1, 1)
                        fake_vals = imp_norm[mask == 0].reshape(-1, 1)
                        fid_val   = compute_fid(real_vals, fake_vals)

                records.append({
                    'dataset'  : ds,
                    'miss_rate': mr,
                    'model'    : mname,
                    'fid_final': fid_val,
                })
    return pd.DataFrame(records)


def plot_final_fid_bars(fid_df, datasets, miss_rates, title_prefix='', save_path=None):
    """
    데이터셋별 서브플롯, 결측률별 그룹 바 차트로 3모델 최종 FID 비교.
    """
    COLORS = {'GAIN': 'steelblue', 'WGAN_FWAL': 'seagreen', 'KD': 'darkorange'}
    models = ['GAIN', 'WGAN_FWAL', 'KD']
    n_ds   = len(datasets)
    n_mr   = len(miss_rates)
    x      = np.arange(n_mr)
    width  = 0.25

    fig, axes = plt.subplots(1, n_ds, figsize=(5 * n_ds, 4), sharey=False)
    if n_ds == 1:
        axes = [axes]

    for ax, ds in zip(axes, datasets):
        sub = fid_df[fid_df['dataset'] == ds]
        for i, model in enumerate(models):
            msub = sub[sub['model'] == model].sort_values('miss_rate')
            vals = [msub[msub['miss_rate'] == mr]['fid_final'].values[0]
                    if not msub[msub['miss_rate'] == mr].empty else np.nan
                    for mr in miss_rates]
            bars = ax.bar(x + i * width, vals, width,
                          label=model, color=COLORS[model], alpha=0.85)
            for bar, v in zip(bars, vals):
                if not np.isnan(v):
                    ax.text(bar.get_x() + bar.get_width() / 2,
                            bar.get_height() * 1.02,
                            f'{v:.3f}', ha='center', va='bottom', fontsize=7)

        ax.set_title(ds, fontsize=10, fontweight='bold')
        ax.set_xticks(x + width)
        ax.set_xticklabels([f'{mr}' for mr in miss_rates], fontsize=8)
        ax.set_xlabel('Missing Rate')
        ax.set_ylabel('Final FID ↓')
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3, axis='y')

    plt.suptitle(f'{title_prefix} — Final FID 3모델 비교',
                 fontsize=12, fontweight='bold', y=1.02)
    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path) or '.', exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()


# Light DB 최종 FID
print('\n[Final FID] Light DB — 3모델 비교')
fid_final_df_light = compute_final_fid_all(
    trained_G_all, fid_hist_all, 'light', DATASETS, MISS_RATES)
plot_final_fid_bars(
    fid_final_df_light, DATASETS, MISS_RATES,
    title_prefix='Light DB',
    save_path=os.path.join(ADV_SAVE_DIR, 'light', 'final_fid_bars.png')
)

# Heavy DB 최종 FID
if trained_G_all['heavy']:
    print('\n[Final FID] Heavy DB — 3모델 비교')
    heavy_ds_list = list(trained_G_all['heavy'].keys())
    fid_final_df_heavy = compute_final_fid_all(
        trained_G_all, fid_hist_all, 'heavy', heavy_ds_list, HEAVY_MISS_RATES)
    plot_final_fid_bars(
        fid_final_df_heavy, heavy_ds_list, HEAVY_MISS_RATES,
        title_prefix='Heavy DB',
        save_path=os.path.join(ADV_SAVE_DIR, 'heavy', 'final_fid_bars.png')
    )